In [5]:
# 05_aml_baseline_chrono — Cell 1: setup + verify AML source
import sys
from pathlib import Path

REPO = Path("C:/m1/repo")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import duckdb
import pandas as pd
import numpy as np

from src.config import PATHS, SEED, DUCKDB_FILE, duckdb_connect, Timer

TRAIN_CUTOFF = pd.Timestamp("2022-09-06 13:36:00")

con = duckdb_connect()

print("repo:", PATHS["repo"])
print("work:", PATHS["work"])
print("lake:", PATHS["lake"])
print("DuckDB:", duckdb.__version__)
print("SEED:", SEED)
print("TRAIN_CUTOFF:", TRAIN_CUTOFF)
print("DUCKDB_FILE:", DUCKDB_FILE, "| exists:", DUCKDB_FILE.exists())

WORK_PARQUET = PATHS["work"] / "aml_hismall_clean.parquet"
LAKE_PARQUET = PATHS["lake"] / "aml_hismall_clean.parquet"
print("\nwork parquet:", WORK_PARQUET, "| exists:", WORK_PARQUET.exists())
print("lake parquet:", LAKE_PARQUET, "| exists:", LAKE_PARQUET.exists())

tables = con.execute("SHOW TABLES").fetchdf()
print("\nTables in DUCKDB_FILE:")
print(tables)

if "aml" in tables["name"].values:
    print("\naml schema:")
    print(con.execute("DESCRIBE aml").fetchdf())
    print("\naml row count:", con.execute("SELECT COUNT(*) FROM aml").fetchone()[0])

repo: C:\m1\repo
work: C:\m1\work
lake: D:\m1\lake
DuckDB: 1.5.5
SEED: 42
TRAIN_CUTOFF: 2022-09-06 13:36:00
DUCKDB_FILE: C:\m1\work\m1.duckdb | exists: True

work parquet: C:\m1\work\aml_hismall_clean.parquet | exists: True
lake parquet: D:\m1\lake\aml_hismall_clean.parquet | exists: True

Tables in DUCKDB_FILE:
  name
0  aml

aml schema:
      column_name column_type null   key default extra
0              ts   TIMESTAMP  YES  None    None  None
1       from_bank     VARCHAR  YES  None    None  None
2       from_acct     VARCHAR  YES  None    None  None
3         to_bank     VARCHAR  YES  None    None  None
4         to_acct     VARCHAR  YES  None    None  None
5    amt_received      DOUBLE  YES  None    None  None
6        recv_ccy     VARCHAR  YES  None    None  None
7        amt_paid      DOUBLE  YES  None    None  None
8         pay_ccy     VARCHAR  YES  None    None  None
9      pay_format     VARCHAR  YES  None    None  None
10  is_laundering     TINYINT  YES  None    None  None

In [8]:
# 05_aml_baseline_chrono — Cell 2: FX rate calibration (train-only) + stability check
import numpy as np
import pandas as pd

with Timer("FX: pull train cross-currency pairs"):
    fx_train = con.execute(f"""
        SELECT ts, pay_ccy, recv_ccy, amt_paid, amt_received
        FROM aml
        WHERE ts < TIMESTAMP '{TRAIN_CUTOFF}'
          AND pay_ccy <> recv_ccy
          AND amt_paid > 0 AND amt_received > 0
    """).fetchdf()

print("cross-ccy train rows:", len(fx_train))
print(fx_train[["pay_ccy", "recv_ccy"]].drop_duplicates().shape[0], "distinct ccy pairs")
print("ts range:", fx_train["ts"].min(), "to", fx_train["ts"].max())

ANCHOR_CCY = "US Dollar"

def solve_fx_rates(df):
    """Least squares on log(amt_received/amt_paid) = log(u_pay) - log(u_recv), anchor US Dollar=1."""
    currencies = sorted(set(df["pay_ccy"]) | set(df["recv_ccy"]))
    idx = {c: i for i, c in enumerate(currencies)}
    n = len(currencies)

    y = np.log(df["amt_received"].values / df["amt_paid"].values)
    A = np.zeros((len(df), n))
    for row_i, (p, r) in enumerate(zip(df["pay_ccy"], df["recv_ccy"])):
        A[row_i, idx[p]] += 1.0
        A[row_i, idx[r]] -= 1.0

    assert ANCHOR_CCY in idx, f"{ANCHOR_CCY} not present in currency set"
    anchor_col = idx[ANCHOR_CCY]
    cols = [c for c in range(n) if c != anchor_col]
    A_reduced = A[:, cols]

    log_u_reduced, *_ = np.linalg.lstsq(A_reduced, y, rcond=None)

    log_u = np.zeros(n)
    for k, c in enumerate(cols):
        log_u[c] = log_u_reduced[k]

    return {currencies[i]: float(np.exp(log_u[i])) for i in range(n)}, currencies

rates_full, currencies_full = solve_fx_rates(fx_train)

day1_start = pd.Timestamp("2022-09-01 00:00:00")
day1_end = pd.Timestamp("2022-09-02 00:00:00")
fx_day1 = fx_train[(fx_train["ts"] >= day1_start) & (fx_train["ts"] < day1_end)]
print("\nday-1 cross-ccy rows:", len(fx_day1))

rates_day1, currencies_day1 = solve_fx_rates(fx_day1)

common = sorted(set(currencies_full) & set(currencies_day1))
missing_in_day1 = sorted(set(currencies_full) - set(currencies_day1))

print("\ncurrencies in full train :", len(currencies_full), currencies_full)
print("currencies in day-1      :", len(currencies_day1), currencies_day1)
if missing_in_day1:
    print("MISSING from day-1 (cannot check stability for these):", missing_in_day1)

rows = []
for c in common:
    full_r = rates_full[c]
    d1_r = rates_day1[c]
    pct_diff = abs(d1_r - full_r) / full_r * 100 if full_r != 0 else np.nan
    rows.append({"currency": c, "rate_full_train": full_r, "rate_day1": d1_r, "pct_diff": pct_diff})

stability_df = pd.DataFrame(rows).sort_values("pct_diff", ascending=False)
print("\nFX stability (day-1 vs full-train), sorted by %% diff:")
print(stability_df.to_string(index=False))

max_diff = stability_df["pct_diff"].max()
print(f"\nMax %% diff: {max_diff:.4f}%")
if max_diff < 1.0 and not missing_in_day1:
    print("STABLE -> proceed with train-period cross-currency FX rates (single fixed rate per currency).")
    FX_MODE = "cross_currency"
else:
    print("UNSTABLE or missing coverage -> FALLBACK per #9a: log(amt_paid) + pay_ccy, no cross-currency sums.")
    FX_MODE = "fallback_no_fx"

print("\nFX_MODE =", FX_MODE)

[FX: pull train cross-currency pairs] wall = 0.1 s | RSS now = 0.22 GB | peak RSS = 0.23 GB
cross-ccy train rows: 39749
193 distinct ccy pairs
ts range: 2022-09-01 00:00:00 to 2022-09-06 13:35:00

day-1 cross-ccy rows: 9572

currencies in full train : 15 ['Australian Dollar', 'Bitcoin', 'Brazil Real', 'Canadian Dollar', 'Euro', 'Mexican Peso', 'Ruble', 'Rupee', 'Saudi Riyal', 'Shekel', 'Swiss Franc', 'UK Pound', 'US Dollar', 'Yen', 'Yuan']
currencies in day-1      : 15 ['Australian Dollar', 'Bitcoin', 'Brazil Real', 'Canadian Dollar', 'Euro', 'Mexican Peso', 'Ruble', 'Rupee', 'Saudi Riyal', 'Shekel', 'Swiss Franc', 'UK Pound', 'US Dollar', 'Yen', 'Yuan']

FX stability (day-1 vs full-train), sorted by %% diff:
         currency  rate_full_train    rate_day1  pct_diff
          Bitcoin     11740.741019 11702.313977  0.327297
      Saudi Riyal         0.266354     0.265637  0.269040
     Mexican Peso         0.047394     0.047511  0.246412
            Ruble         0.012860     0.012882  

In [9]:
# 05_aml_baseline_chrono — Cell 3: apply FX rates, create amt_usd columns, persist aml_fx table
import pandas as pd

assert FX_MODE == "cross_currency"
assert set(rates_full.keys()) == {
    'Australian Dollar','Bitcoin','Brazil Real','Canadian Dollar','Euro','Mexican Peso',
    'Ruble','Rupee','Saudi Riyal','Shekel','Swiss Franc','UK Pound','US Dollar','Yen','Yuan'
}, "rate coverage mismatch"

rates_df = pd.DataFrame(
    [{"ccy": c, "rate_to_usd": r} for c, r in rates_full.items()]
)
print(rates_df.sort_values("ccy").to_string(index=False))

con.execute("DROP TABLE IF EXISTS fx_rates")
con.register("rates_df_view", rates_df)
con.execute("CREATE TABLE fx_rates AS SELECT * FROM rates_df_view")

with Timer("build aml_fx (amt_paid_usd, amt_received_usd)"):
    con.execute("""
        DROP TABLE IF EXISTS aml_fx;
        CREATE TABLE aml_fx AS
        SELECT
            a.*,
            a.amt_paid     * fp.rate_to_usd AS amt_paid_usd,
            a.amt_received * fr.rate_to_usd AS amt_received_usd,
            (a.pay_ccy <> a.recv_ccy) AS is_cross_ccy
        FROM aml a
        JOIN fx_rates fp ON a.pay_ccy  = fp.ccy
        JOIN fx_rates fr ON a.recv_ccy = fr.ccy
    """)

n_orig = con.execute("SELECT COUNT(*) FROM aml").fetchone()[0]
n_fx = con.execute("SELECT COUNT(*) FROM aml_fx").fetchone()[0]
print(f"\naml rows: {n_orig} | aml_fx rows: {n_fx} | match: {n_orig == n_fx}")

print("\naml_fx schema:")
print(con.execute("DESCRIBE aml_fx").fetchdf())

print("\nsanity: amt_paid_usd vs amt_received_usd, same-currency rows (should be ~equal, no FX applied):")
print(con.execute("""
    SELECT amt_paid, amt_paid_usd, amt_received, amt_received_usd, pay_ccy, recv_ccy
    FROM aml_fx WHERE pay_ccy = recv_ccy AND pay_ccy = 'US Dollar' LIMIT 3
""").fetchdf())

print("\nsanity: cross-currency row example:")
print(con.execute("""
    SELECT amt_paid, amt_paid_usd, pay_ccy, amt_received, amt_received_usd, recv_ccy
    FROM aml_fx WHERE is_cross_ccy = true LIMIT 3
""").fetchdf())

              ccy  rate_to_usd
Australian Dollar     0.707678
          Bitcoin 11740.741019
      Brazil Real     0.177083
  Canadian Dollar     0.757974
             Euro     1.171777
     Mexican Peso     0.047394
            Ruble     0.012860
            Rupee     0.013616
      Saudi Riyal     0.266354
           Shekel     0.296163
      Swiss Franc     1.092991
         UK Pound     1.291278
        US Dollar     1.000000
              Yen     0.009494
             Yuan     0.149452


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[build aml_fx (amt_paid_usd, amt_received_usd)] wall = 5.5 s | RSS now = 0.76 GB | peak RSS = 1.60 GB

aml rows: 5078345 | aml_fx rows: 5078345 | match: True

aml_fx schema:
         column_name column_type null   key default extra
0                 ts   TIMESTAMP  YES  None    None  None
1          from_bank     VARCHAR  YES  None    None  None
2          from_acct     VARCHAR  YES  None    None  None
3            to_bank     VARCHAR  YES  None    None  None
4            to_acct     VARCHAR  YES  None    None  None
5       amt_received      DOUBLE  YES  None    None  None
6           recv_ccy     VARCHAR  YES  None    None  None
7           amt_paid      DOUBLE  YES  None    None  None
8            pay_ccy     VARCHAR  YES  None    None  None
9         pay_format     VARCHAR  YES  None    None  None
10     is_laundering     TINYINT  YES  None    None  None
11               src     VARCHAR  YES  None    None  None
12               dst     VARCHAR  YES  None    None  None
13      amt_pa

In [10]:
# 05_aml_baseline_chrono — Cell 4a: sender out_/in_ cnt+sum, windows 1h/24h/7d (12 features)
import numpy as np
import pandas as pd

# --- deterministic id, guarded against re-run ---
if "id" not in con.execute("DESCRIBE aml_fx").fetchdf()["column_name"].values:
    with Timer("add deterministic id to aml_fx"):
        con.execute("""
            CREATE OR REPLACE TABLE aml_fx AS
            SELECT ROW_NUMBER() OVER (ORDER BY ts, src, dst, amt_paid) AS id, *
            FROM aml_fx
        """)
print("aml_fx columns:", list(con.execute("SELECT * FROM aml_fx LIMIT 0").fetchdf().columns))

# --- per-(account, ts) rollups, self-loops excluded, then cumulative ---
with Timer("build cum_out, cum_in"):
    con.execute("""
        CREATE OR REPLACE TABLE out_per_minute AS
        SELECT src AS account, ts, COUNT(*) AS cnt, SUM(amt_paid_usd) AS amt_sum
        FROM aml_fx
        WHERE src <> dst
        GROUP BY src, ts
    """)
    con.execute("""
        CREATE OR REPLACE TABLE cum_out AS
        SELECT account, ts,
            SUM(cnt) OVER (PARTITION BY account ORDER BY ts
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_cnt,
            SUM(amt_sum) OVER (PARTITION BY account ORDER BY ts
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_sum
        FROM out_per_minute
    """)
    con.execute("""
        CREATE OR REPLACE TABLE in_per_minute AS
        SELECT dst AS account, ts, COUNT(*) AS cnt, SUM(amt_received_usd) AS amt_sum
        FROM aml_fx
        WHERE src <> dst
        GROUP BY dst, ts
    """)
    con.execute("""
        CREATE OR REPLACE TABLE cum_in AS
        SELECT account, ts,
            SUM(cnt) OVER (PARTITION BY account ORDER BY ts
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_cnt,
            SUM(amt_sum) OVER (PARTITION BY account ORDER BY ts
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_sum
        FROM in_per_minute
    """)

print("cum_out rows:", con.execute("SELECT COUNT(*) FROM cum_out").fetchone()[0])
print("cum_in rows :", con.execute("SELECT COUNT(*) FROM cum_in").fetchone()[0])

# --- sender_features via ASOF joins ---
with Timer("sender_features (ASOF)"):
    con.execute("""
        CREATE OR REPLACE TABLE sender_features AS
        WITH t AS (
            SELECT id, src, ts,
                   ts - INTERVAL '1 minute' AS upper_x,
                   ts - INTERVAL '1 hour'   - INTERVAL '1 minute' AS lower_1h,
                   ts - INTERVAL '24 hours' - INTERVAL '1 minute' AS lower_24h,
                   ts - INTERVAL '7 days'   - INTERVAL '1 minute' AS lower_7d
            FROM aml_fx
        )
        SELECT
            t.id,
            COALESCE(ou.cum_cnt,0)  - COALESCE(ol1.cum_cnt,0)  AS out_cnt_1h,
            COALESCE(ou.cum_cnt,0)  - COALESCE(ol24.cum_cnt,0) AS out_cnt_24h,
            COALESCE(ou.cum_cnt,0)  - COALESCE(ol7.cum_cnt,0)  AS out_cnt_7d,
            COALESCE(ou.cum_sum,0)  - COALESCE(ol1.cum_sum,0)  AS out_sum_usd_1h,
            COALESCE(ou.cum_sum,0)  - COALESCE(ol24.cum_sum,0) AS out_sum_usd_24h,
            COALESCE(ou.cum_sum,0)  - COALESCE(ol7.cum_sum,0)  AS out_sum_usd_7d,
            COALESCE(iu.cum_cnt,0)  - COALESCE(il1.cum_cnt,0)  AS in_cnt_1h,
            COALESCE(iu.cum_cnt,0)  - COALESCE(il24.cum_cnt,0) AS in_cnt_24h,
            COALESCE(iu.cum_cnt,0)  - COALESCE(il7.cum_cnt,0)  AS in_cnt_7d,
            COALESCE(iu.cum_sum,0)  - COALESCE(il1.cum_sum,0)  AS in_sum_usd_1h,
            COALESCE(iu.cum_sum,0)  - COALESCE(il24.cum_sum,0) AS in_sum_usd_24h,
            COALESCE(iu.cum_sum,0)  - COALESCE(il7.cum_sum,0)  AS in_sum_usd_7d
        FROM t
        ASOF LEFT JOIN cum_out ou   ON t.src = ou.account   AND ou.ts   <= t.upper_x
        ASOF LEFT JOIN cum_out ol1  ON t.src = ol1.account  AND ol1.ts  <= t.lower_1h
        ASOF LEFT JOIN cum_out ol24 ON t.src = ol24.account AND ol24.ts <= t.lower_24h
        ASOF LEFT JOIN cum_out ol7  ON t.src = ol7.account  AND ol7.ts  <= t.lower_7d
        ASOF LEFT JOIN cum_in  iu   ON t.src = iu.account   AND iu.ts   <= t.upper_x
        ASOF LEFT JOIN cum_in  il1  ON t.src = il1.account  AND il1.ts  <= t.lower_1h
        ASOF LEFT JOIN cum_in  il24 ON t.src = il24.account AND il24.ts <= t.lower_24h
        ASOF LEFT JOIN cum_in  il7  ON t.src = il7.account  AND il7.ts  <= t.lower_7d
    """)

n_sf = con.execute("SELECT COUNT(*) FROM sender_features").fetchone()[0]
n_aml = con.execute("SELECT COUNT(*) FROM aml_fx").fetchone()[0]
print(f"\nsender_features rows: {n_sf} | aml_fx rows: {n_aml} | match: {n_sf == n_aml}")
print("\nsender_features columns:", list(con.execute("SELECT * FROM sender_features LIMIT 0").fetchdf().columns))
print(con.execute("SELECT * FROM sender_features LIMIT 5").fetchdf())

# --- monotonicity assert ---
bad = con.execute("""
    SELECT COUNT(*) FROM sender_features
    WHERE out_cnt_1h > out_cnt_24h OR out_cnt_24h > out_cnt_7d
       OR in_cnt_1h  > in_cnt_24h  OR in_cnt_24h  > in_cnt_7d
       OR out_sum_usd_1h > out_sum_usd_24h + 1e-9 OR out_sum_usd_24h > out_sum_usd_7d + 1e-9
       OR in_sum_usd_1h  > in_sum_usd_24h  + 1e-9 OR in_sum_usd_24h  > in_sum_usd_7d  + 1e-9
""").fetchone()[0]
print(f"\nmonotonicity violations: {bad}")
assert bad == 0, f"{bad} rows violate monotonicity"

# --- first-minute invariant ---
first_minute = con.execute("SELECT MIN(ts) FROM aml_fx").fetchone()[0]
n_first = con.execute(f"SELECT COUNT(*) FROM aml_fx WHERE ts = TIMESTAMP '{first_minute}'").fetchone()[0]
n_zero = con.execute(f"""
    SELECT COUNT(*) FROM aml_fx a JOIN sender_features sf ON a.id = sf.id
    WHERE a.ts = TIMESTAMP '{first_minute}'
      AND sf.out_cnt_1h = 0 AND sf.out_cnt_24h = 0 AND sf.out_cnt_7d = 0
      AND sf.in_cnt_1h = 0 AND sf.in_cnt_24h = 0 AND sf.in_cnt_7d = 0
""").fetchone()[0]
print(f"\nfirst-minute rows: {n_first} | all-zero sender counts: {n_zero} | match: {n_first == n_zero}")
assert n_first == n_zero, "first-minute invariant FAILED"

# --- brute-force check: stratified sample (first tx + 4 random), self-loops excluded ---
first_tx_id = con.execute("SELECT id FROM aml_fx ORDER BY ts, id LIMIT 1").fetchone()[0]
random_ids = con.execute(f"""
    SELECT id FROM aml_fx USING SAMPLE 4 (bernoulli, 1%) REPEATABLE (42) LIMIT 4
""").fetchdf()["id"].tolist()
sample_ids = [first_tx_id] + random_ids
print("\nsample ids (first=cold-start, rest=random):", sample_ids)

mismatches = []
for tx_id in sample_ids:
    row = con.execute(f"SELECT id, src, ts FROM aml_fx WHERE id = {tx_id}").fetchdf().iloc[0]
    computed = con.execute(f"SELECT * FROM sender_features WHERE id = {tx_id}").fetchdf().iloc[0]

    for label, hours in [("1h", 1), ("24h", 24), ("7d", 168)]:
        bf_out = con.execute(f"""
            SELECT COUNT(*) AS cnt, COALESCE(SUM(amt_paid_usd),0) AS out_sum
            FROM aml_fx
            WHERE src = '{row.src}' AND src <> dst
              AND ts >= TIMESTAMP '{row.ts}' - INTERVAL '{hours} hours'
              AND ts <  TIMESTAMP '{row.ts}'
        """).fetchdf().iloc[0]
        bf_in = con.execute(f"""
            SELECT COUNT(*) AS cnt, COALESCE(SUM(amt_received_usd),0) AS in_sum
            FROM aml_fx
            WHERE dst = '{row.src}' AND src <> dst
              AND ts >= TIMESTAMP '{row.ts}' - INTERVAL '{hours} hours'
              AND ts <  TIMESTAMP '{row.ts}'
        """).fetchdf().iloc[0]

        out_cnt_col, out_sum_col = f"out_cnt_{label}", f"out_sum_usd_{label}"
        in_cnt_col, in_sum_col = f"in_cnt_{label}", f"in_sum_usd_{label}"

        ok_out_cnt = bf_out["cnt"] == computed[out_cnt_col]
        ok_out_sum = abs(bf_out["out_sum"] - computed[out_sum_col]) < 1e-6
        ok_in_cnt = bf_in["cnt"] == computed[in_cnt_col]
        ok_in_sum = abs(bf_in["in_sum"] - computed[in_sum_col]) < 1e-6

        if not (ok_out_cnt and ok_out_sum and ok_in_cnt and ok_in_sum):
            mismatches.append((tx_id, label, dict(
                bf_out_cnt=bf_out["cnt"], computed_out_cnt=computed[out_cnt_col],
                bf_out_sum=bf_out["out_sum"], computed_out_sum=computed[out_sum_col],
                bf_in_cnt=bf_in["cnt"], computed_in_cnt=computed[in_cnt_col],
                bf_in_sum=bf_in["in_sum"], computed_in_sum=computed[in_sum_col],
            )))

print(f"\nbrute-force check: {len(sample_ids)} rows x 3 windows = {len(sample_ids)*3} checks")
print("mismatches:", len(mismatches))
for m in mismatches:
    print(m)

assert len(mismatches) == 0, "Cell 4a brute-force verification FAILED"
print("\nCell 4a: 12 features built and verified (self-loops excluded, monotonicity holds).")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[add deterministic id to aml_fx] wall = 16.8 s | RSS now = 0.76 GB | peak RSS = 3.28 GB
aml_fx columns: ['id', 'ts', 'from_bank', 'from_acct', 'to_bank', 'to_acct', 'amt_received', 'recv_ccy', 'amt_paid', 'pay_ccy', 'pay_format', 'is_laundering', 'src', 'dst', 'amt_paid_usd', 'amt_received_usd', 'is_cross_ccy']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[build cum_out, cum_in] wall = 8.0 s | RSS now = 1.09 GB | peak RSS = 3.28 GB
cum_out rows: 4075227
cum_in rows : 4364273


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[sender_features (ASOF)] wall = 43.9 s | RSS now = 0.23 GB | peak RSS = 9.06 GB

sender_features rows: 5078345 | aml_fx rows: 5078345 | match: True

sender_features columns: ['id', 'out_cnt_1h', 'out_cnt_24h', 'out_cnt_7d', 'out_sum_usd_1h', 'out_sum_usd_24h', 'out_sum_usd_7d', 'in_cnt_1h', 'in_cnt_24h', 'in_cnt_7d', 'in_sum_usd_1h', 'in_sum_usd_24h', 'in_sum_usd_7d']
        id  out_cnt_1h  out_cnt_24h  out_cnt_7d  out_sum_usd_1h  \
0  3900783         0.0          9.0        50.0        0.000000   
1  3953674         0.0         10.0        49.0        0.000000   
2  3958216         1.0         11.0        50.0    77674.886162   
3  3958544         2.0         12.0        51.0   782537.356652   
4  3960469         3.0         13.0        52.0   817710.060458   

   out_sum_usd_24h  out_sum_usd_7d  in_cnt_1h  in_cnt_24h  in_cnt_7d  \
0     1.737340e+06    8.690317e+06        0.0         3.0       20.0   
1     1.737572e+06    8.689433e+06        1.0         4.0       21.0   
2     1.81

ParserException: Parser Error: syntax error at or near "%"

LINE 2:     SELECT id FROM aml_fx USING SAMPLE 4 (bernoulli, 1%) REPEATABLE (42) LIMIT 4
                                                              ^

In [11]:
# brute-force check: stratified sample (first tx + 4 random), self-loops excluded
import numpy as np
np.random.seed(42)

total = con.execute("SELECT COUNT(*) FROM aml_fx").fetchone()[0]
sample_offsets = sorted(np.random.choice(total, size=4, replace=False).tolist())
random_ids = [
    con.execute(f"SELECT id FROM aml_fx ORDER BY id LIMIT 1 OFFSET {o}").fetchone()[0]
    for o in sample_offsets
]
print("sample offsets:", sample_offsets)

first_tx_id = con.execute("SELECT id FROM aml_fx ORDER BY ts, id LIMIT 1").fetchone()[0]
sample_ids = [first_tx_id] + random_ids
print("sample ids (first=cold-start, rest=random):", sample_ids)

mismatches = []
for tx_id in sample_ids:
    row = con.execute(f"SELECT id, src, ts FROM aml_fx WHERE id = {tx_id}").fetchdf().iloc[0]
    computed = con.execute(f"SELECT * FROM sender_features WHERE id = {tx_id}").fetchdf().iloc[0]

    for label, hours in [("1h", 1), ("24h", 24), ("7d", 168)]:
        bf_out = con.execute(f"""
            SELECT COUNT(*) AS cnt, COALESCE(SUM(amt_paid_usd),0) AS out_sum
            FROM aml_fx
            WHERE src = '{row.src}' AND src <> dst
              AND ts >= TIMESTAMP '{row.ts}' - INTERVAL '{hours} hours'
              AND ts <  TIMESTAMP '{row.ts}'
        """).fetchdf().iloc[0]
        bf_in = con.execute(f"""
            SELECT COUNT(*) AS cnt, COALESCE(SUM(amt_received_usd),0) AS in_sum
            FROM aml_fx
            WHERE dst = '{row.src}' AND src <> dst
              AND ts >= TIMESTAMP '{row.ts}' - INTERVAL '{hours} hours'
              AND ts <  TIMESTAMP '{row.ts}'
        """).fetchdf().iloc[0]

        out_cnt_col, out_sum_col = f"out_cnt_{label}", f"out_sum_usd_{label}"
        in_cnt_col, in_sum_col = f"in_cnt_{label}", f"in_sum_usd_{label}"

        ok_out_cnt = bf_out["cnt"] == computed[out_cnt_col]
        ok_out_sum = abs(bf_out["out_sum"] - computed[out_sum_col]) < 1e-6
        ok_in_cnt = bf_in["cnt"] == computed[in_cnt_col]
        ok_in_sum = abs(bf_in["in_sum"] - computed[in_sum_col]) < 1e-6

        if not (ok_out_cnt and ok_out_sum and ok_in_cnt and ok_in_sum):
            mismatches.append((tx_id, label, dict(
                bf_out_cnt=bf_out["cnt"], computed_out_cnt=computed[out_cnt_col],
                bf_out_sum=bf_out["out_sum"], computed_out_sum=computed[out_sum_col],
                bf_in_cnt=bf_in["cnt"], computed_in_cnt=computed[in_cnt_col],
                bf_in_sum=bf_in["in_sum"], computed_in_sum=computed[in_sum_col],
            )))

print(f"\nbrute-force check: {len(sample_ids)} rows x 3 windows = {len(sample_ids)*3} checks")
print("mismatches:", len(mismatches))
for m in mismatches:
    print(m)

assert len(mismatches) == 0, "Cell 4a brute-force verification FAILED"
print("\nCell 4a: 12 features built and verified (self-loops excluded, monotonicity holds).")

sample offsets: [298872, 405190, 746726, 1388703]
sample ids (first=cold-start, rest=random): [1, 298873, 405191, 746727, 1388704]

brute-force check: 5 rows x 3 windows = 15 checks
mismatches: 0

Cell 4a: 12 features built and verified (self-loops excluded, monotonicity holds).


In [12]:
import gc
gc.collect()

355

In [17]:
# 05_aml_baseline_chrono — Cell 4b-1: cheap derived features
import numpy as np
import pandas as pd

with Timer("mins_since_last_inflow (ASOF, all-time)"):
    con.execute("""
        CREATE OR REPLACE TABLE last_inflow AS
        WITH t AS (SELECT id, src, ts FROM aml_fx)
        SELECT
            t.id,
            DATEDIFF('minute', h.last_in_ts, t.ts) AS mins_since_last_inflow
        FROM t
        ASOF LEFT JOIN (
            SELECT dst AS account, ts AS last_in_ts
            FROM aml_fx WHERE src <> dst
        ) h
        ON t.src = h.account AND h.last_in_ts <= t.ts - INTERVAL '1 minute'
    """)
    print("last_inflow rows:", con.execute("SELECT COUNT(*) FROM last_inflow").fetchone()[0])

with Timer("sender_derived_cheap"):
    con.execute("""
        CREATE OR REPLACE TABLE sender_derived_cheap AS
        SELECT
            sf.id,
            CASE WHEN sf.in_sum_usd_24h = 0 THEN NULL
                 ELSE sf.out_sum_usd_24h / sf.in_sum_usd_24h END AS pass_through_24h,
            li.mins_since_last_inflow,
            CASE WHEN sf.in_sum_usd_24h = 0 THEN NULL
                 ELSE a.amt_paid_usd / sf.in_sum_usd_24h END AS amt_over_inflow_24h
        FROM sender_features sf
        JOIN aml_fx a ON a.id = sf.id
        JOIN last_inflow li ON li.id = sf.id
    """)
    print("sender_derived_cheap rows:", con.execute("SELECT COUNT(*) FROM sender_derived_cheap").fetchone()[0])
    print(con.execute("SELECT * FROM sender_derived_cheap LIMIT 5").fetchdf())

n_null_pt = con.execute("SELECT COUNT(*) FROM sender_derived_cheap WHERE pass_through_24h IS NULL").fetchone()[0]
n_null_aoi = con.execute("SELECT COUNT(*) FROM sender_derived_cheap WHERE amt_over_inflow_24h IS NULL").fetchone()[0]
n_zero_insum = con.execute("SELECT COUNT(*) FROM sender_features WHERE in_sum_usd_24h = 0").fetchone()[0]
print(f"NULL pass_through: {n_null_pt} | NULL amt_over_inflow: {n_null_aoi} | zero in_sum: {n_zero_insum}")
assert n_null_pt == n_zero_insum and n_null_aoi == n_zero_insum
print("\nCell 4b-1: 3 features built. Total: 15/34.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

last_inflow rows: 5078345
[mins_since_last_inflow (ASOF, all-time)] wall = 5.2 s | RSS now = 0.34 GB | peak RSS = 9.39 GB
sender_derived_cheap rows: 5078345
        id  pass_through_24h  mins_since_last_inflow  amt_over_inflow_24h
0  3900783        450.952224                    1020             0.060413
1  3953674        353.006517                       3            15.780489
2  3958216        237.900392                      12            92.376983
3  3958544        330.277376                      13             4.609620
4  3960469        331.630700                       5             6.420832
[sender_derived_cheap] wall = 1.0 s | RSS now = 0.85 GB | peak RSS = 9.39 GB
NULL pass_through: 2395001 | NULL amt_over_inflow: 2395001 | zero in_sum: 2395001

Cell 4b-1: 3 features built. Total: 15/34.


In [20]:
# 05_aml_baseline_chrono — Cell 4b-2: new-counterparty rate 24h (DECISION #9c), 2 features
# Single LAG per (src,dst), projected to both directions. Supersedes the two-LAG version.

with Timer("pair_events: LAG once, tie-break by id"):
    con.execute("""
        CREATE OR REPLACE TABLE pair_events AS
        SELECT id, src, dst, ts,
               LAG(ts) OVER (PARTITION BY src, dst ORDER BY ts, id) AS prev_pair_ts
        FROM aml_fx
        WHERE src <> dst
    """)
    con.execute("""
        CREATE OR REPLACE TABLE pair_events_flagged AS
        SELECT id, src, dst, ts,
               (prev_pair_ts IS NULL OR ts - prev_pair_ts > INTERVAL '24 hours') AS is_new_pair
        FROM pair_events
    """)

n_total = con.execute("SELECT COUNT(*) FROM pair_events_flagged").fetchone()[0]
n_new = con.execute("SELECT COUNT(*) FROM pair_events_flagged WHERE is_new_pair").fetchone()[0]
print(f"pair_events: {n_total:,} rows | new: {n_new:,} ({100*n_new/n_total:.2f}%)")

con.execute("DROP TABLE pair_events")

with Timer("cumulative new-pair + ASOF"):
    con.execute("""
        CREATE OR REPLACE TABLE out_new_per_minute AS
        SELECT src AS account, ts, COUNT(*) FILTER (WHERE is_new_pair) AS new_cnt
        FROM pair_events_flagged GROUP BY src, ts
    """)
    con.execute("""
        CREATE OR REPLACE TABLE cum_out_new AS
        SELECT account, ts,
               SUM(new_cnt) OVER (PARTITION BY account ORDER BY ts
                                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_new
        FROM out_new_per_minute
    """)
    con.execute("""
        CREATE OR REPLACE TABLE in_new_per_minute AS
        SELECT dst AS account, ts, COUNT(*) FILTER (WHERE is_new_pair) AS new_cnt
        FROM pair_events_flagged GROUP BY dst, ts
    """)
    con.execute("""
        CREATE OR REPLACE TABLE cum_in_new AS
        SELECT account, ts,
               SUM(new_cnt) OVER (PARTITION BY account ORDER BY ts
                                  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_new
        FROM in_new_per_minute
    """)
    con.execute("""
        CREATE OR REPLACE TABLE sender_new_counterparty_24h AS
        WITH t AS (
            SELECT id, src, ts,
                   ts - INTERVAL '1 minute' AS upper_x,
                   ts - INTERVAL '24 hours' - INTERVAL '1 minute' AS lower_24h
            FROM aml_fx
        )
        SELECT
            t.id,
            COALESCE(ou.cum_new,0) - COALESCE(ol.cum_new,0) AS n_new_dst_24h,
            COALESCE(iu.cum_new,0) - COALESCE(il.cum_new,0) AS n_new_senders_in_24h
        FROM t
        ASOF LEFT JOIN cum_out_new ou ON t.src = ou.account AND ou.ts <= t.upper_x
        ASOF LEFT JOIN cum_out_new ol ON t.src = ol.account AND ol.ts <= t.lower_24h
        ASOF LEFT JOIN cum_in_new  iu ON t.src = iu.account AND iu.ts <= t.upper_x
        ASOF LEFT JOIN cum_in_new  il ON t.src = il.account AND il.ts <= t.lower_24h
    """)

n_snc = con.execute("SELECT COUNT(*) FROM sender_new_counterparty_24h").fetchone()[0]
assert n_snc == n_aml
print(f"sender_new_counterparty_24h rows: {n_snc} | match: {n_snc == n_aml}")
print(con.execute("SELECT * FROM sender_new_counterparty_24h LIMIT 5").fetchdf())

bad_out = con.execute("""
    SELECT COUNT(*) FROM sender_new_counterparty_24h snc
    JOIN sender_features sf ON sf.id = snc.id
    WHERE snc.n_new_dst_24h > sf.out_cnt_24h
""").fetchone()[0]
bad_in = con.execute("""
    SELECT COUNT(*) FROM sender_new_counterparty_24h snc
    JOIN sender_features sf ON sf.id = snc.id
    WHERE snc.n_new_senders_in_24h > sf.in_cnt_24h
""").fetchone()[0]
print(f"n_new_dst > out_cnt violations: {bad_out} | n_new_senders_in > in_cnt: {bad_in}")
assert bad_out == 0 and bad_in == 0

mismatches_4b2 = []
for tx_id in sample_ids:
    row = con.execute(f"SELECT id, src, ts FROM aml_fx WHERE id = {tx_id}").fetchdf().iloc[0]
    computed = con.execute(f"SELECT * FROM sender_new_counterparty_24h WHERE id = {tx_id}").fetchdf().iloc[0]

    bf_out_new = con.execute(f"""
        WITH window_tx AS (
            SELECT dst, ts FROM aml_fx
            WHERE src = '{row.src}' AND src <> dst
              AND ts >= TIMESTAMP '{row.ts}' - INTERVAL '24 hours' AND ts < TIMESTAMP '{row.ts}'
        ), with_prev AS (
            SELECT w.dst, w.ts,
                (SELECT MAX(ts) FROM aml_fx p WHERE p.src = '{row.src}' AND p.dst = w.dst AND p.ts < w.ts) AS prev_ts
            FROM window_tx w
        )
        SELECT COUNT(*) AS cnt FROM with_prev
        WHERE prev_ts IS NULL OR ts - prev_ts > INTERVAL '24 hours'
    """).fetchone()[0]
    bf_in_new = con.execute(f"""
        WITH window_tx AS (
            SELECT src, ts FROM aml_fx
            WHERE dst = '{row.src}' AND src <> dst
              AND ts >= TIMESTAMP '{row.ts}' - INTERVAL '24 hours' AND ts < TIMESTAMP '{row.ts}'
        ), with_prev AS (
            SELECT w.src, w.ts,
                (SELECT MAX(ts) FROM aml_fx p WHERE p.dst = '{row.src}' AND p.src = w.src AND p.ts < w.ts) AS prev_ts
            FROM window_tx w
        )
        SELECT COUNT(*) AS cnt FROM with_prev
        WHERE prev_ts IS NULL OR ts - prev_ts > INTERVAL '24 hours'
    """).fetchone()[0]

    ok_out = bf_out_new == computed["n_new_dst_24h"]
    ok_in  = bf_in_new  == computed["n_new_senders_in_24h"]
    if not (ok_out and ok_in):
        mismatches_4b2.append((tx_id, dict(
            bf_out=bf_out_new, db_out=int(computed["n_new_dst_24h"]),
            bf_in=bf_in_new,  db_in=int(computed["n_new_senders_in_24h"]))))

print(f"\nCell 4b-2 brute-force: {len(sample_ids)} rows | mismatches: {len(mismatches_4b2)}")
for m in mismatches_4b2:
    print(m)
assert len(mismatches_4b2) == 0

print("\nCell 4b-2 done: 2 features. Total: 17/34.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[pair_events: LAG once, tie-break by id] wall = 7.7 s | RSS now = 1.15 GB | peak RSS = 9.39 GB
pair_events: 4,487,133 rows | new: 1,613,573 (35.96%)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[cumulative new-pair + ASOF] wall = 44.7 s | RSS now = 1.18 GB | peak RSS = 9.39 GB
sender_new_counterparty_24h rows: 5078345 | match: True
        id  n_new_dst_24h  n_new_senders_in_24h
0  1531800            2.0                   0.0
1  1538542            2.0                   0.0
2  1539042            2.0                   0.0
3  1540567            2.0                   0.0
4  1549307            2.0                   0.0
n_new_dst > out_cnt violations: 0 | n_new_senders_in > in_cnt: 0

Cell 4b-2 brute-force: 5 rows | mismatches: 0

Cell 4b-2 done: 2 features. Total: 17/34.


In [23]:
# 05_aml_baseline_chrono — Cell 4c-audit: same-minute repeats of a (src,dst) pair (#9d point 8)
n_groups, n_rows = con.execute("""
    SELECT COUNT(*), COALESCE(SUM(c), 0) FROM (
        SELECT src, dst, ts, COUNT(*) AS c FROM aml_fx
        WHERE src <> dst GROUP BY src, dst, ts HAVING COUNT(*) > 1)
""").fetchone()
n_pe = con.execute("SELECT COUNT(*) FROM aml_fx WHERE src <> dst").fetchone()[0]
print(f"(src,dst,minute) groups with >1 transfer: {n_groups:,} | rows in them: {n_rows:,} "
      f"({100*n_rows/n_pe:.2f}% of {n_pe:,} non-self-loop rows)")

(src,dst,minute) groups with >1 transfer: 96,324 | rows in them: 193,725 (4.32% of 4,487,133 non-self-loop rows)


In [24]:
# 05_aml_baseline_chrono — Cell 4c-0: FX estimator check, per-pair medians vs fitted rates (#9d point 5)
import pandas as pd

assert "fx_train" in globals() and "solve_fx_rates" in globals(), "re-run Cell 2 first"

pair_med = (fx_train.assign(ratio=fx_train["amt_received"] / fx_train["amt_paid"])
            .groupby(["pay_ccy", "recv_ccy"], as_index=False)["ratio"].median())
print("currency pairs:", len(pair_med))

# solve_fx_rates reads log(amt_received / amt_paid); with amt_paid = 1 this is log(median ratio)
pair_med = pair_med.rename(columns={"ratio": "amt_received"}).assign(amt_paid=1.0)
rates_med, _ = solve_fx_rates(pair_med)

cmp = pd.DataFrame({"fitted": pd.Series(rates_full), "median_based": pd.Series(rates_med)})
cmp["pct_diff"] = (cmp["median_based"] - cmp["fitted"]).abs() / cmp["fitted"] * 100
print(cmp.sort_values("pct_diff", ascending=False).to_string())

max_d = cmp["pct_diff"].max()
print(f"\nmax % diff (median-based vs fitted): {max_d:.4f}%")
assert max_d < 1.0, "median-based rates differ by >= 1% -> STOP, review per #9d point 5"
print("Fitted rates kept (#9d point 5). aml_fx unchanged.")

currency pairs: 193
                         fitted  median_based  pct_diff
Bitcoin            11740.741019  11830.793703  0.767010
Shekel                 0.296163      0.293894  0.766263
Mexican Peso           0.047394      0.047611  0.457381
Yuan                   0.149452      0.149685  0.155795
Saudi Riyal            0.266354      0.266706  0.132195
Yen                    0.009494      0.009488  0.059461
Rupee                  0.013616      0.013624  0.057507
Brazil Real            0.177083      0.177179  0.054213
Euro                   1.171777      1.172303  0.044907
Australian Dollar      0.707678      0.707867  0.026591
UK Pound               1.291278      1.291520  0.018732
Ruble                  0.012860      0.012858  0.014691
Swiss Franc            1.092991      1.092942  0.004528
Canadian Dollar        0.757974      0.757998  0.003206
US Dollar              1.000000      1.000000  0.000000

max % diff (median-based vs fitted): 0.7670%
Fitted rates kept (#9d point 5). aml_f

In [25]:
# 05_aml_baseline_chrono — Cell 4c-1: receiver-side features (DECISION #9, #9a, #9c, #9d), 8 features
# Reuses cum_in, cum_out (4a) and cum_in_new (4b-2); ASOF joins on t.dst instead of t.src.
# Sums are NULL when dst has no earlier inflow at all (#9d point 4).

with Timer("receiver_features (ASOF on dst)"):
    con.execute("""
        CREATE OR REPLACE TABLE receiver_features AS
        WITH t AS (
            SELECT id, dst, ts,
                   ts - INTERVAL '1 minute' AS upper_x,
                   ts - INTERVAL '1 hour'   - INTERVAL '1 minute' AS lower_1h,
                   ts - INTERVAL '24 hours' - INTERVAL '1 minute' AS lower_24h,
                   ts - INTERVAL '7 days'   - INTERVAL '1 minute' AS lower_7d
            FROM aml_fx
        )
        SELECT
            t.id,
            COALESCE(iu.cum_cnt,0) - COALESCE(il1.cum_cnt,0)  AS dst_in_cnt_1h,
            COALESCE(iu.cum_cnt,0) - COALESCE(il24.cum_cnt,0) AS dst_in_cnt_24h,
            COALESCE(iu.cum_cnt,0) - COALESCE(il7.cum_cnt,0)  AS dst_in_cnt_7d,
            iu.cum_sum - COALESCE(il1.cum_sum,0)  AS dst_in_sum_usd_1h,
            iu.cum_sum - COALESCE(il24.cum_sum,0) AS dst_in_sum_usd_24h,
            iu.cum_sum - COALESCE(il7.cum_sum,0)  AS dst_in_sum_usd_7d,
            COALESCE(ou.cum_cnt,0) - COALESCE(ol24.cum_cnt,0) AS dst_out_cnt_24h,
            COALESCE(nu.cum_new,0) - COALESCE(nl.cum_new,0)   AS dst_new_counterparty_in_24h
        FROM t
        ASOF LEFT JOIN cum_in     iu   ON t.dst = iu.account   AND iu.ts   <= t.upper_x
        ASOF LEFT JOIN cum_in     il1  ON t.dst = il1.account  AND il1.ts  <= t.lower_1h
        ASOF LEFT JOIN cum_in     il24 ON t.dst = il24.account AND il24.ts <= t.lower_24h
        ASOF LEFT JOIN cum_in     il7  ON t.dst = il7.account  AND il7.ts  <= t.lower_7d
        ASOF LEFT JOIN cum_out    ou   ON t.dst = ou.account   AND ou.ts   <= t.upper_x
        ASOF LEFT JOIN cum_out    ol24 ON t.dst = ol24.account AND ol24.ts <= t.lower_24h
        ASOF LEFT JOIN cum_in_new nu   ON t.dst = nu.account   AND nu.ts   <= t.upper_x
        ASOF LEFT JOIN cum_in_new nl   ON t.dst = nl.account   AND nl.ts   <= t.lower_24h
    """)

n_rf = con.execute("SELECT COUNT(*) FROM receiver_features").fetchone()[0]
print(f"receiver_features rows: {n_rf} | aml_fx rows: {n_aml} | match: {n_rf == n_aml}")
assert n_rf == n_aml
print(con.execute("SELECT * FROM receiver_features LIMIT 5").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[receiver_features (ASOF on dst)] wall = 201.3 s | RSS now = 0.28 GB | peak RSS = 9.39 GB
receiver_features rows: 5078345 | aml_fx rows: 5078345 | match: True
        id  dst_in_cnt_1h  dst_in_cnt_24h  dst_in_cnt_7d  dst_in_sum_usd_1h  \
0  2360210            0.0             0.0           10.0           0.000000   
1  2360209            0.0             0.0           10.0           0.000000   
2  2362848            2.0             2.0           12.0        5581.544644   
3  2364744            3.0             3.0           13.0       10104.800548   
4  3017363            0.0             0.0           14.0           0.000000   

   dst_in_sum_usd_24h  dst_in_sum_usd_7d  dst_out_cnt_24h  \
0            0.000000       45140.362470              0.0   
1            0.000000       45140.362470              0.0   
2         5581.544644       50721.907114              0.0   
3        10104.800548       55245.163019              0.0   
4            0.000000       59633.019242              7.0   


In [26]:
# 05_aml_baseline_chrono — Cell 4c-2: invariants for receiver_features

bad_mono = con.execute("""
    SELECT COUNT(*) FROM receiver_features
    WHERE dst_in_cnt_1h > dst_in_cnt_24h OR dst_in_cnt_24h > dst_in_cnt_7d
       OR dst_in_sum_usd_1h > dst_in_sum_usd_24h + 1e-6 OR dst_in_sum_usd_24h > dst_in_sum_usd_7d + 1e-6
""").fetchone()[0]
bad_new = con.execute("""
    SELECT COUNT(*) FROM receiver_features WHERE dst_new_counterparty_in_24h > dst_in_cnt_24h
""").fetchone()[0]
bad_null = con.execute("""
    SELECT COUNT(*) FROM receiver_features
    WHERE (dst_in_sum_usd_1h IS NULL) <> (dst_in_sum_usd_7d IS NULL)
       OR (dst_in_sum_usd_24h IS NULL) <> (dst_in_sum_usd_7d IS NULL)
       OR (dst_in_sum_usd_7d IS NULL AND dst_in_cnt_7d > 0)
""").fetchone()[0]
bad_cons = con.execute("""
    SELECT COUNT(*) FROM receiver_features
    WHERE (dst_in_cnt_7d = 0 AND dst_in_sum_usd_7d IS NOT NULL AND dst_in_sum_usd_7d <> 0)
       OR (dst_in_cnt_7d > 0 AND (dst_in_sum_usd_7d IS NULL OR dst_in_sum_usd_7d = 0))
""").fetchone()[0]
n_null = con.execute("SELECT COUNT(*) FROM receiver_features WHERE dst_in_sum_usd_7d IS NULL").fetchone()[0]
print(f"monotonicity: {bad_mono} | new > in_cnt_24h: {bad_new} | NULL-pattern: {bad_null} | count<->sum: {bad_cons}")
print(f"cold-start NULL sums: {n_null:,} ({100*n_null/n_aml:.2f}% of rows)")
assert bad_mono == 0 and bad_new == 0 and bad_null == 0 and bad_cons == 0

first_minute = con.execute("SELECT MIN(ts) FROM aml_fx").fetchone()[0]
n_first, n_ok = con.execute(f"""
    SELECT COUNT(*),
           COUNT(*) FILTER (WHERE rf.dst_in_cnt_7d = 0 AND rf.dst_out_cnt_24h = 0
                              AND rf.dst_new_counterparty_in_24h = 0 AND rf.dst_in_sum_usd_7d IS NULL)
    FROM aml_fx a JOIN receiver_features rf ON a.id = rf.id
    WHERE a.ts = TIMESTAMP '{first_minute}'
""").fetchone()
print(f"first-minute rows: {n_first} | zero counts + NULL sums: {n_ok} | match: {n_first == n_ok}")
assert n_first == n_ok

monotonicity: 0 | new > in_cnt_24h: 0 | NULL-pattern: 0 | count<->sum: 0
cold-start NULL sums: 713,911 (14.06% of rows)
first-minute rows: 10977 | zero counts + NULL sums: 10977 | match: True


In [27]:
# 05_aml_baseline_chrono — Cell 4c-3: brute-force check on sample_ids (first tx + 4 random)
import pandas as pd

def close(a, b):
    return abs(a - b) <= 1e-6 * max(1.0, abs(b))

mismatches_4c = []
for tx_id in sample_ids:
    dst, ts = con.execute(f"SELECT dst, ts FROM aml_fx WHERE id = {tx_id}").fetchone()
    c = con.execute(f"SELECT * FROM receiver_features WHERE id = {tx_id}").fetchdf().iloc[0]
    base = f"FROM aml_fx WHERE dst = '{dst}' AND src <> dst AND ts < TIMESTAMP '{ts}'"
    n_hist = con.execute(f"SELECT COUNT(*) {base}").fetchone()[0]

    for label, h in [("1h", 1), ("24h", 24), ("7d", 168)]:
        cnt, s = con.execute(f"SELECT COUNT(*), SUM(amt_received_usd) {base} "
                             f"AND ts >= TIMESTAMP '{ts}' - INTERVAL '{h} hours'").fetchone()
        got = c[f"dst_in_sum_usd_{label}"]
        s_val = 0.0 if s is None else float(s)
        ok_sum = pd.isna(got) if n_hist == 0 else (not pd.isna(got) and close(got, s_val))
        if cnt != c[f"dst_in_cnt_{label}"] or not ok_sum:
            mismatches_4c.append((tx_id, label, cnt, c[f"dst_in_cnt_{label}"], s, got))

    out24 = con.execute(f"""SELECT COUNT(*) FROM aml_fx WHERE src = '{dst}' AND src <> dst
        AND ts >= TIMESTAMP '{ts}' - INTERVAL '24 hours' AND ts < TIMESTAMP '{ts}'""").fetchone()[0]
    new24 = con.execute(f"""
        WITH w AS (SELECT id, src, ts FROM aml_fx WHERE dst = '{dst}' AND src <> dst
                   AND ts >= TIMESTAMP '{ts}' - INTERVAL '24 hours' AND ts < TIMESTAMP '{ts}')
        SELECT COUNT(*) FROM w
        WHERE NOT EXISTS (SELECT 1 FROM aml_fx p WHERE p.dst = '{dst}' AND p.src = w.src
                          AND (p.ts < w.ts OR (p.ts = w.ts AND p.id < w.id))
                          AND p.ts >= w.ts - INTERVAL '24 hours')""").fetchone()[0]
    if out24 != c["dst_out_cnt_24h"]:
        mismatches_4c.append((tx_id, "out_24h", out24, c["dst_out_cnt_24h"]))
    if new24 != c["dst_new_counterparty_in_24h"]:
        mismatches_4c.append((tx_id, "new_24h", new24, c["dst_new_counterparty_in_24h"]))

print(f"Cell 4c brute-force: {len(sample_ids)} rows | mismatches: {len(mismatches_4c)}")
for m in mismatches_4c:
    print(m)
assert len(mismatches_4c) == 0
print("\nCell 4c done: 8 receiver features. Total: 25/34.")

Cell 4c brute-force: 5 rows | mismatches: 0

Cell 4c done: 8 receiver features. Total: 25/34.


In [29]:
# 05 — state snapshot for continuing without Claude Code
print("tables:")
for (t,) in con.execute("SELECT table_name FROM information_schema.tables ORDER BY 1").fetchall():
    cols = [c for (c,) in con.execute(
        f"SELECT column_name FROM information_schema.columns WHERE table_name='{t}' ORDER BY ordinal_position").fetchall()]
    n = con.execute(f"SELECT count(*) FROM {t}").fetchone()[0]
    print(f"- {t} ({n:,} rows): {cols}")
print("\npython vars:", sorted(k for k, v in globals().items()
      if not k.startswith("_") and type(v).__name__ in ("str", "int", "float", "Timestamp", "datetime", "dict", "list")))

tables:
- aml (5,078,345 rows): ['ts', 'from_bank', 'from_acct', 'to_bank', 'to_acct', 'amt_received', 'recv_ccy', 'amt_paid', 'pay_ccy', 'pay_format', 'is_laundering', 'src', 'dst']
- aml_fx (5,078,345 rows): ['id', 'ts', 'from_bank', 'from_acct', 'to_bank', 'to_acct', 'amt_received', 'recv_ccy', 'amt_paid', 'pay_ccy', 'pay_format', 'is_laundering', 'src', 'dst', 'amt_paid_usd', 'amt_received_usd', 'is_cross_ccy']
- cum_in (4,364,273 rows): ['account', 'ts', 'cum_cnt', 'cum_sum']
- cum_in_new (4,364,273 rows): ['account', 'ts', 'cum_new']
- cum_out (4,075,227 rows): ['account', 'ts', 'cum_cnt', 'cum_sum']
- cum_out_new (4,075,227 rows): ['account', 'ts', 'cum_new']
- fx_rates (15 rows): ['ccy', 'rate_to_usd']
- in_new_per_minute (4,364,273 rows): ['account', 'ts', 'new_cnt']
- in_pair_gap (4,487,133 rows): ['id', 'src', 'dst', 'ts', 'prev_pair_ts']
- in_pair_new (4,487,133 rows): ['id', 'dst', 'ts', 'is_new_pair']
- in_per_minute (4,364,273 rows): ['account', 'ts', 'cnt', 'amt_sum']
-

In [30]:
# 05 — Cell 4d: pair_cnt_7d and observed ages; self-loops excluded (#9 p3, #9a naming)
with Timer("4d build"):
    con.execute("""CREATE OR REPLACE TABLE pair_cum AS
        SELECT src, dst, ts, sum(cnt) OVER (PARTITION BY src, dst ORDER BY ts ROWS UNBOUNDED PRECEDING) AS cum
        FROM (SELECT src, dst, ts, count(*) AS cnt FROM pair_events_flagged GROUP BY ALL)""")
    con.execute("""CREATE OR REPLACE TABLE first_seen AS
        SELECT acct, min(ts) AS first_ts FROM (
            SELECT src AS acct, ts FROM pair_events_flagged
            UNION ALL SELECT dst, ts FROM pair_events_flagged) GROUP BY 1""")
    con.execute("""CREATE OR REPLACE TABLE pair_age_features AS
        SELECT a.id,
               CASE WHEN a.src = a.dst THEN 0
                    ELSE coalesce(c1.cum, 0) - coalesce(c2.cum, 0) END AS pair_cnt_7d,  -- window [t-7d, t)
               CASE WHEN fs.first_ts < a.ts THEN date_diff('minute', fs.first_ts, a.ts) / 60.0 END AS src_observed_age_hours,
               CASE WHEN fd.first_ts < a.ts THEN date_diff('minute', fd.first_ts, a.ts) / 60.0 END AS dst_observed_age_hours
        FROM aml_fx a
        ASOF LEFT JOIN pair_cum c1 ON a.src = c1.src AND a.dst = c1.dst AND a.ts > c1.ts
        ASOF LEFT JOIN pair_cum c2 ON a.src = c2.src AND a.dst = c2.dst AND a.ts - INTERVAL 7 DAY > c2.ts
        LEFT JOIN first_seen fs ON fs.acct = a.src
        LEFT JOIN first_seen fd ON fd.acct = a.dst""")
    con.execute("DROP TABLE pair_cum")

n4d = con.execute("SELECT count(*) FROM pair_age_features").fetchone()[0]
print("rows:", n4d, "| match:", n4d == n_aml)
print(con.execute("""SELECT count(*) FILTER (WHERE pair_cnt_7d < 0) AS neg_pair,
       count(*) FILTER (WHERE p.pair_cnt_7d > s.out_cnt_7d) AS pair_gt_out,
       count(*) FILTER (WHERE p.pair_cnt_7d > r.dst_in_cnt_7d) AS pair_gt_dst_in,
       count(*) FILTER (WHERE src_observed_age_hours < 0 OR dst_observed_age_hours < 0) AS neg_age,
       round(100.0 * avg((src_observed_age_hours IS NULL)::INT), 2) AS src_age_null_pct
    FROM pair_age_features p JOIN sender_features s USING (id) JOIN receiver_features r USING (id)""").df().T.to_string(header=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[4d build] wall = 22.9 s | RSS now = 0.60 GB | peak RSS = 9.39 GB
rows: 5078345 | match: True
neg_pair           0.00
pair_gt_out        0.00
pair_gt_dst_in     0.00
neg_age            0.00
src_age_null_pct  11.58


In [31]:
# 05 — Cell 4e: transaction-level features known at t (#9 p4)
with Timer("4e build"):
    con.execute("""CREATE OR REPLACE TABLE txn_features AS
        SELECT id,
               ln(amt_paid_usd)          AS log_amount_usd,   -- amounts > 0 (audit)
               pay_format, pay_ccy,
               is_cross_ccy::TINYINT     AS is_cross_ccy,
               hour(ts)                  AS hour_of_day,
               (src = dst)::TINYINT      AS is_self_loop
        FROM aml_fx""")
print(con.execute("""SELECT count(*) AS n, count(*) FILTER (WHERE log_amount_usd IS NULL) AS null_amt,
       sum(is_self_loop) AS self_loops, sum(is_cross_ccy) AS cross_ccy,
       min(hour_of_day) AS h_min, max(hour_of_day) AS h_max FROM txn_features""").df().T.to_string(header=False))
print("FX_MODE:", FX_MODE)

[4e build] wall = 2.5 s | RSS now = 0.72 GB | peak RSS = 9.39 GB
n           5078345.0
null_amt          0.0
self_loops   591212.0
cross_ccy     72170.0
h_min             0.0
h_max            23.0
FX_MODE: cross_currency


In [32]:
# 05 — Cell 4d-check: brute-force recomputation from strictly earlier non-self-loop rows
ids = con.execute("SELECT id FROM aml_fx WHERE src <> dst USING SAMPLE 20 ROWS (reservoir, 42)").fetchall()
bad = 0
for (i,) in ids:
    src_, dst_, t_ = con.execute("SELECT src, dst, ts FROM aml_fx WHERE id = ?", [i]).fetchone()
    pc = con.execute("""SELECT count(*) FROM pair_events_flagged WHERE src=? AND dst=?
                        AND ts >= ?::TIMESTAMP - INTERVAL 7 DAY AND ts < ?""", [src_, dst_, t_, t_]).fetchone()[0]
    ages = []
    for acct in (src_, dst_):
        f = con.execute("""SELECT min(ts) FROM pair_events_flagged
                           WHERE (src=? OR dst=?) AND ts < ?""", [acct, acct, t_]).fetchone()[0]
        ages.append(None if f is None else (t_ - f).total_seconds() / 3600)
    got = con.execute("SELECT pair_cnt_7d, src_observed_age_hours, dst_observed_age_hours "
                      "FROM pair_age_features WHERE id = ?", [i]).fetchone()
    ok = got[0] == pc and all((a is None and g is None) or (a is not None and g is not None and abs(a - g) < 1e-6)
                              for a, g in zip(ages, got[1:]))
    bad += not ok
print(f"4d brute-force: {len(ids)} rows | mismatches: {bad}")

4d brute-force: 4 rows | mismatches: 0


In [36]:
# 05 — Cell 4f: assemble 34 features, split labels, checks, save to work/ (fixed EXCLUDE list)
FEATURE_TABLES = ["sender_features", "sender_derived_cheap", "sender_new_counterparty_24h",
                  "receiver_features", "pair_age_features", "txn_features"]
joins = " ".join(f"JOIN {t} USING (id)" for t in FEATURE_TABLES)
with Timer("4f assemble"):
    con.execute(f"""CREATE OR REPLACE TABLE aml_features AS
        SELECT a.id, a.ts, a.is_laundering,
               CASE WHEN a.ts < TIMESTAMP '2022-09-06 13:36' THEN 'train'
                    WHEN a.ts < TIMESTAMP '2022-09-08 16:12' THEN 'valid' ELSE 'test' END AS split,
               (a.ts >= TIMESTAMP '2022-09-08 16:12' AND a.ts < TIMESTAMP '2022-09-11') AS in_test_core,
               a.pay_format, a.pay_ccy, a.is_cross_ccy,
               * EXCLUDE (id, ts, is_laundering, pay_format, pay_ccy, is_cross_ccy)
        FROM aml_fx a {joins}""")

META = ["id", "ts", "is_laundering", "split", "in_test_core"]
FEATS = [c for (c,) in con.execute("SELECT column_name FROM information_schema.columns "
         "WHERE table_name='aml_features' ORDER BY ordinal_position").fetchall()
         if (c not in META and c not in con.execute("SELECT * FROM aml_fx LIMIT 0").df().columns)
         or c in ("pay_format", "pay_ccy", "is_cross_ccy")]
print(len(FEATS), "features:", FEATS)
assert len(FEATS) == 34, len(FEATS)

print(con.execute("""SELECT split, count(*) n, sum(is_laundering) n_laund,
       sum(in_test_core::INT) core FROM aml_features GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

cnt_cols = [c for c in FEATS if "cnt" in c or c.startswith("n_new")]
first = con.execute(f"""SELECT count(*) FILTER (WHERE {' OR '.join(f'{c} <> 0' for c in cnt_cols)})
       FROM aml_features WHERE ts = (SELECT min(ts) FROM aml_features)""").fetchone()[0]
print("first-minute rows with non-zero counts:", first)

# sender-side cold-start NULL check (in addition to receiver-side already checked)
print(con.execute("""SELECT CAST(ts AS DATE) d,
       round(100*avg((in_sum_usd_24h IS NULL)::INT),1) null_in24_sender,
       round(100*avg((dst_in_sum_usd_24h IS NULL)::INT),1) null_in24_receiver,
       round(100*avg((src_observed_age_hours IS NULL)::INT),1) null_age
       FROM aml_features WHERE split='train' GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

out = PATHS["work"] / "aml_features.parquet"
with Timer("save parquet"):
    con.execute(f"COPY (SELECT * FROM aml_features ORDER BY ts, id) TO '{out.as_posix()}' (FORMAT parquet, COMPRESSION zstd)")
print(f"saved {out}  {out.stat().st_size/1e6:.1f} MB")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[4f assemble] wall = 30.0 s | RSS now = 1.99 GB | peak RSS = 9.39 GB
34 features: ['pay_format', 'pay_ccy', 'is_cross_ccy', 'out_cnt_1h', 'out_cnt_24h', 'out_cnt_7d', 'out_sum_usd_1h', 'out_sum_usd_24h', 'out_sum_usd_7d', 'in_cnt_1h', 'in_cnt_24h', 'in_cnt_7d', 'in_sum_usd_1h', 'in_sum_usd_24h', 'in_sum_usd_7d', 'pass_through_24h', 'mins_since_last_inflow', 'amt_over_inflow_24h', 'n_new_dst_24h', 'n_new_senders_in_24h', 'dst_in_cnt_1h', 'dst_in_cnt_24h', 'dst_in_cnt_7d', 'dst_in_sum_usd_1h', 'dst_in_sum_usd_24h', 'dst_in_sum_usd_7d', 'dst_out_cnt_24h', 'dst_new_counterparty_in_24h', 'pair_cnt_7d', 'src_observed_age_hours', 'dst_observed_age_hours', 'log_amount_usd', 'hour_of_day', 'is_self_loop']
split       n  n_laund      core
 test 1015882   1798.0 1014774.0
train 3046861   2297.0       0.0
valid 1015602   1082.0       0.0
first-minute rows with non-zero counts: 0
         d  null_in24_sender  null_in24_receiver  null_age
2022-09-01               0.0                56.1      43.7
20

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[save parquet] wall = 17.8 s | RSS now = 2.04 GB | peak RSS = 9.39 GB
saved C:\m1\work\aml_features.parquet  532.5 MB


In [37]:
# Read-only verification: Task 1, 2, 3 — no table modifications
import pandas as pd

print("=" * 60)
print("TASK 1 — LAG partition semantics test")
print("=" * 60)
test_df = pd.DataFrame({
    "src": ["A", "A", "A"],
    "dst": ["X", "Y", "X"],
    "ts": pd.to_datetime(["2022-09-01 00:00", "2022-09-01 00:05", "2022-09-01 00:10"])
})
con.register("test_df", test_df)
result1 = con.execute("""
    SELECT src, dst, ts,
           LAG(ts) OVER (PARTITION BY src, dst ORDER BY ts) AS prev_pair_ts
    FROM test_df ORDER BY ts
""").fetchdf()
print(result1)
row3_prev = result1.iloc[2]["prev_pair_ts"]
row1_ts = result1.iloc[0]["ts"]
print(f"\nRow 3 prev_pair_ts == Row 1 ts? {row3_prev == row1_ts}")

print("\n" + "=" * 60)
print("TASK 2 — quantify old (gap>24h) vs new (ever-seen) definition")
print("=" * 60)

# rebuild pair_events fresh from aml_fx (self-loops excluded), independent of any cached table
con.execute("""
    CREATE OR REPLACE TABLE _verify_pairs AS
    SELECT id, src, dst, ts,
           LAG(ts) OVER (PARTITION BY src, dst ORDER BY ts, id) AS prev_pair_ts,
           ROW_NUMBER() OVER (PARTITION BY src, dst ORDER BY ts, id) AS rn
    FROM aml_fx
    WHERE src <> dst
""")

counts = con.execute("""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN prev_pair_ts IS NULL OR ts - prev_pair_ts > INTERVAL '24 hours' THEN 1 ELSE 0 END) AS n1_old_def,
        SUM(CASE WHEN rn = 1 THEN 1 ELSE 0 END) AS n2_new_def
    FROM _verify_pairs
""").fetchdf().iloc[0]

total = counts["total"]
n1 = counts["n1_old_def"]
n2 = counts["n2_new_def"]
p1 = 100 * n1 / total
p2 = 100 * n2 / total
print(f"total pair-events (self-loops excluded): {total:,}")
print(f"N1 (old def, gap>24h or first): {n1:,} ({p1:.2f}%)")
print(f"N2 (new def, first-ever occurrence): {n2:,} ({p2:.2f}%)")
print(f"difference: {n1-n2:,} rows ({p1-p2:.2f} percentage points)")

con.execute("DROP TABLE _verify_pairs")

print("\n" + "=" * 60)
print("TASK 3 — self-loop / observed_age check")
print("=" * 60)

# find an account whose earliest appearance in aml_fx is a self-loop
first_appear = con.execute("""
    WITH all_appear AS (
        SELECT src AS account, ts, (src = dst) AS is_self FROM aml_fx
        UNION ALL
        SELECT dst AS account, ts, (src = dst) AS is_self FROM aml_fx
    ),
    ranked AS (
        SELECT account, ts, is_self,
               ROW_NUMBER() OVER (PARTITION BY account ORDER BY ts) AS rn
        FROM all_appear
    )
    SELECT account, ts, is_self FROM ranked WHERE rn = 1 AND is_self = true
    LIMIT 1
""").fetchdf()

if len(first_appear) == 0:
    print("No account found whose earliest appearance is a self-loop.")
else:
    acct = first_appear.iloc[0]["account"]
    print(f"Account with self-loop as first appearance: {acct}")
    print(first_appear)
    # check if pair_age_features / src_observed_age_hours table exists and what it shows
    tables = con.execute("SHOW TABLES").fetchdf()["name"].tolist()
    if "pair_age_features" in tables or "pair_ages" in tables:
        tbl = "pair_age_features" if "pair_age_features" in tables else "pair_ages"
        age_check = con.execute(f"""
            SELECT a.id, a.src, a.dst, a.ts, a.is_self_loop,
                   pa.src_observed_age_hours, pa.dst_observed_age_hours
            FROM aml_fx a JOIN {tbl} pa ON pa.id = a.id
            WHERE (a.src = '{acct}' OR a.dst = '{acct}')
            ORDER BY a.ts LIMIT 5
        """).fetchdf()
        print(age_check)
    else:
        print("pair_age_features/pair_ages table not found yet (Cell 4d may not have run).")

print("\nDone. No tables modified except temp _verify_pairs (dropped).")

TASK 1 — LAG partition semantics test
  src dst                  ts prev_pair_ts
0   A   X 2022-09-01 00:00:00          NaT
1   A   Y 2022-09-01 00:05:00          NaT
2   A   X 2022-09-01 00:10:00   2022-09-01

Row 3 prev_pair_ts == Row 1 ts? True

TASK 2 — quantify old (gap>24h) vs new (ever-seen) definition


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

total pair-events (self-loops excluded): 4,487,133.0
N1 (old def, gap>24h or first): 1,613,573.0 (35.96%)
N2 (new def, first-ever occurrence): 647,939.0 (14.44%)
difference: 965,634.0 rows (21.52 percentage points)

TASK 3 — self-loop / observed_age check
Account with self-loop as first appearance: 01729_80066E7B0
           account                  ts  is_self
0  01729_80066E7B0 2022-09-01 00:02:00     True


BinderException: Binder Error: Table "a" does not have a column named "is_self_loop"

Candidate bindings: : "id"

LINE 2:             SELECT a.id, a.src, a.dst, a.ts, a.is_self_loop,
                                                     ^

In [38]:
print([c for c in con.execute("SELECT * FROM aml_fx LIMIT 0").df().columns if 'self' in c.lower()])

[]


In [41]:
# DIAGNOSTIC CELL — read-only, does not modify any table
import pandas as pd

print("=" * 60)
print("TASK 1b — ROW_NUMBER partition test")
print("=" * 60)
test_df = pd.DataFrame({
    "id": [1, 2, 3],
    "src": ["A", "A", "A"],
    "dst": ["X", "Y", "X"],
    "ts": pd.to_datetime(["2022-09-01 00:00", "2022-09-01 00:05", "2022-09-01 00:10"]),
})
con.register("test_df", test_df)
result = con.execute("""
    SELECT src, dst, ts,
           ROW_NUMBER() OVER (PARTITION BY src, dst ORDER BY ts, id) AS rn
    FROM test_df ORDER BY ts
""").fetchdf()
print(result.to_string(index=False))
rn_row3 = int(result.iloc[2]["rn"])
print(f"\nRow 3 (second A->X) rn = {rn_row3}  (expected 2: {'PASS' if rn_row3 == 2 else 'FAIL'})")

print("\n" + "=" * 60)
print("TASK 3 retry — self-loop in observed_age")
print("=" * 60)

# A. Column check
cols = con.execute("SELECT * FROM aml_fx LIMIT 0").df().columns.tolist()
self_cols = [c for c in cols if "self" in c.lower()]
print(f"\nA. Columns on aml_fx containing 'self': {self_cols or 'none'}")

# B. Tables present
tables = con.execute("SHOW TABLES").fetchdf()["name"].tolist()
print(f"\nB. Tables in session: {tables}")

# C. Concrete first-seen comparison for the target account
acc = "01729_80066E7B0"
print(f"\nC. Account {acc} first-seen timestamps:")
res = con.execute(f"""
    SELECT
      (SELECT MIN(ts) FROM aml_fx WHERE src = '{acc}')                       AS first_any_as_src,
      (SELECT MIN(ts) FROM aml_fx WHERE src = '{acc}' AND src <> dst)        AS first_non_self_as_src,
      (SELECT MIN(ts) FROM aml_fx WHERE dst = '{acc}')                       AS first_any_as_dst,
      (SELECT MIN(ts) FROM aml_fx WHERE dst = '{acc}' AND src <> dst)        AS first_non_self_as_dst
""").fetchdf()
print(res.to_string(index=False))

# D. If pair_ages exists, compare stored age vs both computations
for tbl in ["pair_ages", "pair_age_features"]:
    if tbl in tables:
        print(f"\nD. {tbl} rows for account {acc} (first 5):")
        try:
            df = con.execute(f"""
                SELECT a.id, a.ts, a.src, a.dst,
                       pa.src_observed_age_hours AS stored_src_age,
                       EXTRACT(EPOCH FROM (a.ts - (SELECT MIN(ts) FROM aml_fx WHERE src = a.src))) / 3600.0 AS age_any_src,
                       EXTRACT(EPOCH FROM (a.ts - (SELECT MIN(ts) FROM aml_fx WHERE src = a.src AND src <> dst))) / 3600.0 AS age_non_self_src
                FROM aml_fx a JOIN {tbl} pa ON pa.id = a.id
                WHERE a.src = '{acc}'
                ORDER BY a.ts LIMIT 5
            """).fetchdf()
            print(df.to_string(index=False))
            print("\n   Interpretation: stored_src_age matches age_any_src  -> self-loops INCLUDED")
            print("                   stored_src_age matches age_non_self_src -> self-loops EXCLUDED")
        except Exception as e:
            print(f"   query failed: {e}")
        break
else:
    print(f"\nD. No pair_ages table found. Session has: {tables}")
    print("   Cell 4d may need to be re-run before Task 3 can complete.")

TASK 1b — ROW_NUMBER partition test
src dst                  ts  rn
  A   X 2022-09-01 00:00:00   1
  A   Y 2022-09-01 00:05:00   1
  A   X 2022-09-01 00:10:00   2

Row 3 (second A->X) rn = 2  (expected 2: PASS)

TASK 3 retry — self-loop in observed_age

A. Columns on aml_fx containing 'self': none

B. Tables in session: ['aml', 'aml_features', 'aml_fx', 'cum_in', 'cum_in_new', 'cum_out', 'cum_out_new', 'first_seen', 'fx_rates', 'in_new_per_minute', 'in_pair_gap', 'in_pair_new', 'in_per_minute', 'last_inflow', 'out_new_per_minute', 'out_pair_gap', 'out_pair_new', 'out_per_minute', 'pair_age_features', 'pair_events_flagged', 'rates_df_view', 'receiver_features', 'sender_derived_cheap', 'sender_features', 'sender_new_counterparty_24h', 'test_df', 'txn_features']

C. Account 01729_80066E7B0 first-seen timestamps:
   first_any_as_src first_non_self_as_src    first_any_as_dst first_non_self_as_dst
2022-09-01 00:02:00   2022-09-06 03:21:00 2022-09-01 00:02:00                   NaT

D. pair_a

In [48]:
# Cell 4b-2 v2: first-seen counterparty via ROW_NUMBER (#9f), 2 features
# Replaces the LAG-based version. Definition: "first occurrence ever" per (src, dst).

# Clean up prior artifacts to avoid confusion
for tbl in ["out_pair_gap", "in_pair_gap", "out_pair_new", "in_pair_new",
            "pair_events", "pair_events_flagged", "out_new_per_minute",
            "in_new_per_minute", "cum_out_new", "cum_in_new",
            "sender_new_counterparty_24h"]:
    con.execute(f"DROP TABLE IF EXISTS {tbl}")

with Timer("pair_first_seen (ROW_NUMBER)"):
    con.execute("""
        CREATE OR REPLACE TABLE pair_first_seen AS
        SELECT id, src, dst, ts,
               (ROW_NUMBER() OVER (PARTITION BY src, dst ORDER BY ts, id) = 1) AS is_first_seen
        FROM aml_fx
        WHERE src <> dst
    """)

n_total = con.execute("SELECT COUNT(*) FROM pair_first_seen").fetchone()[0]
n_first = con.execute("SELECT COUNT(*) FROM pair_first_seen WHERE is_first_seen").fetchone()[0]
print(f"pair_first_seen: {n_total:,} rows | first-ever: {n_first:,} ({100*n_first/n_total:.2f}%)")
assert n_total == 4_487_133, f"unexpected row count {n_total}"  # matches Task 2 baseline
assert n_first == 647_939, f"unexpected first-ever count {n_first}"  # matches Task 2 new-def count

with Timer("cum_first_seen_out/in + ASOF"):
    con.execute("""
        CREATE OR REPLACE TABLE out_first_seen_per_minute AS
        SELECT src AS account, ts, SUM(CASE WHEN is_first_seen THEN 1 ELSE 0 END) AS first_cnt
        FROM pair_first_seen GROUP BY src, ts
    """)
    con.execute("""
        CREATE OR REPLACE TABLE cum_out_first_seen AS
        SELECT account, ts,
               SUM(first_cnt) OVER (PARTITION BY account ORDER BY ts
                                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_first
        FROM out_first_seen_per_minute
    """)
    con.execute("""
        CREATE OR REPLACE TABLE in_first_seen_per_minute AS
        SELECT dst AS account, ts, SUM(CASE WHEN is_first_seen THEN 1 ELSE 0 END) AS first_cnt
        FROM pair_first_seen GROUP BY dst, ts
    """)
    con.execute("""
        CREATE OR REPLACE TABLE cum_in_first_seen AS
        SELECT account, ts,
               SUM(first_cnt) OVER (PARTITION BY account ORDER BY ts
                                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_first
        FROM in_first_seen_per_minute
    """)
    con.execute("""
        CREATE OR REPLACE TABLE sender_first_seen_24h AS
        WITH t AS (
            SELECT id, src, ts,
                   ts - INTERVAL '1 minute' AS upper_x,
                   ts - INTERVAL '24 hours' - INTERVAL '1 minute' AS lower_24h
            FROM aml_fx
        )
        SELECT
            t.id,
            COALESCE(ou.cum_first,0) - COALESCE(ol.cum_first,0) AS n_first_seen_dst_24h,
            COALESCE(iu.cum_first,0) - COALESCE(il.cum_first,0) AS n_first_seen_senders_in_24h
        FROM t
        ASOF LEFT JOIN cum_out_first_seen ou ON t.src = ou.account AND ou.ts <= t.upper_x
        ASOF LEFT JOIN cum_out_first_seen ol ON t.src = ol.account AND ol.ts <= t.lower_24h
        ASOF LEFT JOIN cum_in_first_seen  iu ON t.src = iu.account AND iu.ts <= t.upper_x
        ASOF LEFT JOIN cum_in_first_seen  il ON t.src = il.account AND il.ts <= t.lower_24h
    """)

n_sfs = con.execute("SELECT COUNT(*) FROM sender_first_seen_24h").fetchone()[0]
assert n_sfs == 5_078_345
print(f"sender_first_seen_24h rows: {n_sfs}")
print(con.execute("SELECT * FROM sender_first_seen_24h LIMIT 5").fetchdf())

# Invariant: first_seen counts must not exceed total counts
bad = con.execute("""
    SELECT COUNT(*) FROM sender_first_seen_24h s
    JOIN sender_features sf ON sf.id = s.id
    WHERE s.n_first_seen_dst_24h        > sf.out_cnt_24h
       OR s.n_first_seen_senders_in_24h > sf.in_cnt_24h
""").fetchone()[0]
print(f"first_seen > total violations: {bad}")
assert bad == 0

# Brute-force on the original 5 sample ids
mism = []
for tx_id in sample_ids:
    row = con.execute(f"SELECT id, src, ts FROM aml_fx WHERE id = {tx_id}").fetchdf().iloc[0]
    c = con.execute(f"SELECT * FROM sender_first_seen_24h WHERE id = {tx_id}").fetchdf().iloc[0]
    bf_out = con.execute(f"""
        WITH w AS (SELECT id, dst, ts FROM aml_fx
                   WHERE src = '{row.src}' AND src <> dst
                     AND ts >= TIMESTAMP '{row.ts}' - INTERVAL '24 hours'
                     AND ts <  TIMESTAMP '{row.ts}')
        SELECT COUNT(*) FROM w
        WHERE NOT EXISTS (
            SELECT 1 FROM aml_fx p
            WHERE p.src = '{row.src}' AND p.dst = w.dst
              AND (p.ts < w.ts OR (p.ts = w.ts AND p.id < w.id))
        )
    """).fetchone()[0]
    bf_in = con.execute(f"""
        WITH w AS (SELECT id, src, ts FROM aml_fx
                   WHERE dst = '{row.src}' AND src <> dst
                     AND ts >= TIMESTAMP '{row.ts}' - INTERVAL '24 hours'
                     AND ts <  TIMESTAMP '{row.ts}')
        SELECT COUNT(*) FROM w
        WHERE NOT EXISTS (
            SELECT 1 FROM aml_fx p
            WHERE p.dst = '{row.src}' AND p.src = w.src
              AND (p.ts < w.ts OR (p.ts = w.ts AND p.id < w.id))
        )
    """).fetchone()[0]
    if bf_out != c["n_first_seen_dst_24h"] or bf_in != c["n_first_seen_senders_in_24h"]:
        mism.append((tx_id, int(bf_out), int(c["n_first_seen_dst_24h"]),
                          int(bf_in), int(c["n_first_seen_senders_in_24h"])))
print(f"brute-force mismatches: {len(mism)}")
for m in mism: print(m)
assert len(mism) == 0
print("\nCell 4b-2 v2 done: 2 features rebuilt. Names: n_first_seen_dst_24h, n_first_seen_senders_in_24h.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[pair_first_seen (ROW_NUMBER)] wall = 6.3 s | RSS now = 2.00 GB | peak RSS = 9.39 GB
pair_first_seen: 4,487,133 rows | first-ever: 647,939 (14.44%)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[cum_first_seen_out/in + ASOF] wall = 59.2 s | RSS now = 2.20 GB | peak RSS = 9.39 GB
sender_first_seen_24h rows: 5078345
        id  n_first_seen_dst_24h  n_first_seen_senders_in_24h
0  3513960                   0.0                          0.0
1  3513963                   0.0                          0.0
2  3513962                   0.0                          0.0
3  3513951                   0.0                          0.0
4  3513957                   0.0                          0.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

first_seen > total violations: 0
brute-force mismatches: 0

Cell 4b-2 v2 done: 2 features rebuilt. Names: n_first_seen_dst_24h, n_first_seen_senders_in_24h.


In [49]:
# Cell 4d v2: pair_cnt_7d + observed_age_hours (self-loop-inclusive first-seen, per #9f)
# Replaces prior pair_age_features (which excluded self-loops from first-seen).

for tbl in ["pair_age_features", "pair_per_minute", "cum_pair",
            "pair_cnt_7d_feat", "account_first_seen", "observed_age_feat"]:
    con.execute(f"DROP TABLE IF EXISTS {tbl}")

# --- Part 1: pair_cnt_7d (self-loops excluded from pair aggregates, per #9 point 3) ---
with Timer("pair_cnt_7d"):
    con.execute("""
        CREATE OR REPLACE TABLE pair_per_minute AS
        SELECT src, dst, ts, COUNT(*) AS cnt
        FROM aml_fx WHERE src <> dst
        GROUP BY src, dst, ts
    """)
    con.execute("""
        CREATE OR REPLACE TABLE cum_pair AS
        SELECT src, dst, ts,
               SUM(cnt) OVER (PARTITION BY src, dst ORDER BY ts
                              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_cnt
        FROM pair_per_minute
    """)
    con.execute("""
        CREATE OR REPLACE TABLE pair_cnt_7d_feat AS
        WITH t AS (
            SELECT id, src, dst, ts,
                   ts - INTERVAL '1 minute' AS upper_x,
                   ts - INTERVAL '7 days' - INTERVAL '1 minute' AS lower_7d
            FROM aml_fx WHERE src <> dst
        )
        SELECT
            t.id,
            COALESCE(u.cum_cnt,0) - COALESCE(l.cum_cnt,0) AS pair_cnt_7d
        FROM t
        ASOF LEFT JOIN cum_pair u ON t.src = u.src AND t.dst = u.dst AND u.ts <= t.upper_x
        ASOF LEFT JOIN cum_pair l ON t.src = l.src AND t.dst = l.dst AND l.ts <= t.lower_7d
    """)

# --- Part 2: observed_age_hours — self-loop-inclusive first-seen (per #9f.3) ---
with Timer("first-seen inclusive of self-loops"):
    con.execute("""
        CREATE OR REPLACE TABLE account_first_seen AS
        SELECT account, MIN(ts) AS first_ts FROM (
            SELECT src AS account, ts FROM aml_fx
            UNION ALL
            SELECT dst AS account, ts FROM aml_fx
        ) GROUP BY account
    """)
    con.execute("""
        CREATE OR REPLACE TABLE observed_age_feat AS
        SELECT
            a.id,
            CASE WHEN afs_s.first_ts < a.ts
                 THEN EXTRACT(EPOCH FROM (a.ts - afs_s.first_ts)) / 3600.0
                 ELSE NULL END AS src_observed_age_hours,
            CASE WHEN afs_d.first_ts < a.ts
                 THEN EXTRACT(EPOCH FROM (a.ts - afs_d.first_ts)) / 3600.0
                 ELSE NULL END AS dst_observed_age_hours
        FROM aml_fx a
        LEFT JOIN account_first_seen afs_s ON afs_s.account = a.src
        LEFT JOIN account_first_seen afs_d ON afs_d.account = a.dst
    """)

# --- Part 3: assemble ---
with Timer("assemble pair_age_features"):
    con.execute("""
        CREATE OR REPLACE TABLE pair_age_features AS
        SELECT
            a.id,
            COALESCE(pc.pair_cnt_7d, 0) AS pair_cnt_7d,
            oa.src_observed_age_hours,
            oa.dst_observed_age_hours
        FROM aml_fx a
        LEFT JOIN pair_cnt_7d_feat pc ON pc.id = a.id
        LEFT JOIN observed_age_feat oa ON oa.id = a.id
    """)

n = con.execute("SELECT COUNT(*) FROM pair_age_features").fetchone()[0]
assert n == 5_078_345, n
print(f"\npair_age_features rows: {n}")
print(con.execute("SELECT * FROM pair_age_features LIMIT 5").fetchdf())

# --- verify on the Task 3 test account ---
print("\n--- Task 3 account verification (01729_80066E7B0) ---")
print(con.execute("""
    SELECT a.id, a.ts, pa.src_observed_age_hours, pa.dst_observed_age_hours, pa.pair_cnt_7d
    FROM aml_fx a JOIN pair_age_features pa ON pa.id = a.id
    WHERE a.src = '01729_80066E7B0'
    ORDER BY a.ts LIMIT 5
""").fetchdf().to_string(index=False))
print("\nExpected: row 1 NULL (first-ever appearance itself);")
print("          row 2 = 0.05 hours (3 min after first self-loop);")
print("          row 3 = ~123.3 hours (5 days later)")

# --- invariants ---
bad_age = con.execute("""
    SELECT COUNT(*) FROM pair_age_features
    WHERE src_observed_age_hours < 0 OR dst_observed_age_hours < 0
""").fetchone()[0]
assert bad_age == 0, bad_age
print(f"\nnegative ages: {bad_age}")

bad_pc = con.execute("""
    SELECT COUNT(*) FROM pair_age_features pa
    JOIN sender_features sf ON sf.id = pa.id
    WHERE pa.pair_cnt_7d > sf.out_cnt_7d
""").fetchone()[0]
assert bad_pc == 0, bad_pc
print(f"pair_cnt_7d > out_cnt_7d violations: {bad_pc}")

# --- first-minute: all ages NULL, pair_cnt = 0 ---
first_minute = con.execute("SELECT MIN(ts) FROM aml_fx").fetchone()[0]
n_first = con.execute(f"SELECT COUNT(*) FROM aml_fx WHERE ts = TIMESTAMP '{first_minute}'").fetchone()[0]
n_ok = con.execute(f"""
    SELECT COUNT(*) FROM aml_fx a JOIN pair_age_features pa ON pa.id = a.id
    WHERE a.ts = TIMESTAMP '{first_minute}'
      AND pa.src_observed_age_hours IS NULL
      AND pa.dst_observed_age_hours IS NULL
      AND pa.pair_cnt_7d = 0
""").fetchone()[0]
print(f"first-minute rows: {n_first} | all NULL age + 0 pair_cnt: {n_ok} | match: {n_first == n_ok}")
assert n_first == n_ok

print("\nCell 4d v2 done: 3 features rebuilt.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[pair_cnt_7d] wall = 32.2 s | RSS now = 2.23 GB | peak RSS = 9.39 GB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[first-seen inclusive of self-loops] wall = 4.0 s | RSS now = 2.74 GB | peak RSS = 9.39 GB
[assemble pair_age_features] wall = 1.9 s | RSS now = 2.81 GB | peak RSS = 9.39 GB

pair_age_features rows: 5078345
        id  pair_cnt_7d  src_observed_age_hours  dst_observed_age_hours
0  1425409          0.0                   33.55               15.366667
1  1425410          3.0                   33.55               33.550000
2  1425411          2.0                   33.55               33.366667
3  1425412          1.0                   33.55               22.500000
4  1425413          1.0                   33.55               33.150000

--- Task 3 account verification (01729_80066E7B0) ---
     id                  ts  src_observed_age_hours  dst_observed_age_hours  pair_cnt_7d
  28308 2022-09-01 00:02:00                     NaN                     NaN          0.0
  61288 2022-09-01 00:05:00                0.050000                0.050000          0.0
2846958 2022-09-06 03:21:00            

In [50]:
# Cell 4c v2: receiver features rebuilt with cum_in_first_seen (#9f)
# Column rename: dst_new_counterparty_in_24h -> dst_first_seen_in_24h

with Timer("receiver_features v2"):
    con.execute("""
        CREATE OR REPLACE TABLE receiver_features AS
        WITH t AS (
            SELECT id, dst, ts,
                   ts - INTERVAL '1 minute' AS upper_x,
                   ts - INTERVAL '1 hour'   - INTERVAL '1 minute' AS lower_1h,
                   ts - INTERVAL '24 hours' - INTERVAL '1 minute' AS lower_24h,
                   ts - INTERVAL '7 days'   - INTERVAL '1 minute' AS lower_7d
            FROM aml_fx
        )
        SELECT
            t.id,
            COALESCE(iu.cum_cnt,0) - COALESCE(il1.cum_cnt,0)  AS dst_in_cnt_1h,
            COALESCE(iu.cum_cnt,0) - COALESCE(il24.cum_cnt,0) AS dst_in_cnt_24h,
            COALESCE(iu.cum_cnt,0) - COALESCE(il7.cum_cnt,0)  AS dst_in_cnt_7d,
            iu.cum_sum - COALESCE(il1.cum_sum,0)  AS dst_in_sum_usd_1h,
            iu.cum_sum - COALESCE(il24.cum_sum,0) AS dst_in_sum_usd_24h,
            iu.cum_sum - COALESCE(il7.cum_sum,0)  AS dst_in_sum_usd_7d,
            COALESCE(ou.cum_cnt,0) - COALESCE(ol24.cum_cnt,0) AS dst_out_cnt_24h,
            COALESCE(nu.cum_first,0) - COALESCE(nl.cum_first,0) AS dst_first_seen_in_24h
        FROM t
        ASOF LEFT JOIN cum_in             iu   ON t.dst = iu.account   AND iu.ts   <= t.upper_x
        ASOF LEFT JOIN cum_in             il1  ON t.dst = il1.account  AND il1.ts  <= t.lower_1h
        ASOF LEFT JOIN cum_in             il24 ON t.dst = il24.account AND il24.ts <= t.lower_24h
        ASOF LEFT JOIN cum_in             il7  ON t.dst = il7.account  AND il7.ts  <= t.lower_7d
        ASOF LEFT JOIN cum_out            ou   ON t.dst = ou.account   AND ou.ts   <= t.upper_x
        ASOF LEFT JOIN cum_out            ol24 ON t.dst = ol24.account AND ol24.ts <= t.lower_24h
        ASOF LEFT JOIN cum_in_first_seen  nu   ON t.dst = nu.account   AND nu.ts   <= t.upper_x
        ASOF LEFT JOIN cum_in_first_seen  nl   ON t.dst = nl.account   AND nl.ts   <= t.lower_24h
    """)

n_rf = con.execute("SELECT COUNT(*) FROM receiver_features").fetchone()[0]
assert n_rf == 5_078_345, n_rf
print(f"receiver_features rows: {n_rf}")
print("columns:", list(con.execute("SELECT * FROM receiver_features LIMIT 0").df().columns))
print(con.execute("SELECT * FROM receiver_features LIMIT 3").fetchdf().to_string(index=False))

# --- invariants ---
bad_mono = con.execute("""
    SELECT COUNT(*) FROM receiver_features
    WHERE dst_in_cnt_1h > dst_in_cnt_24h OR dst_in_cnt_24h > dst_in_cnt_7d
       OR dst_in_sum_usd_1h > dst_in_sum_usd_24h + 1e-6
       OR dst_in_sum_usd_24h > dst_in_sum_usd_7d + 1e-6
""").fetchone()[0]
assert bad_mono == 0, bad_mono

bad_new = con.execute("""
    SELECT COUNT(*) FROM receiver_features
    WHERE dst_first_seen_in_24h > dst_in_cnt_24h
""").fetchone()[0]
assert bad_new == 0, bad_new

bad_null = con.execute("""
    SELECT COUNT(*) FROM receiver_features
    WHERE (dst_in_sum_usd_1h IS NULL) <> (dst_in_sum_usd_7d IS NULL)
       OR (dst_in_sum_usd_24h IS NULL) <> (dst_in_sum_usd_7d IS NULL)
       OR (dst_in_sum_usd_7d IS NULL AND dst_in_cnt_7d > 0)
""").fetchone()[0]
assert bad_null == 0, bad_null

n_null = con.execute("SELECT COUNT(*) FROM receiver_features WHERE dst_in_sum_usd_7d IS NULL").fetchone()[0]
print(f"\nmonotonicity: {bad_mono} | new>cnt: {bad_new} | NULL-pattern: {bad_null}")
print(f"cold-start NULL sums (dst_in_sum_usd_7d): {n_null:,} ({100*n_null/5_078_345:.2f}%)")

# --- first-minute ---
first_minute = con.execute("SELECT MIN(ts) FROM aml_fx").fetchone()[0]
n_first = con.execute(f"SELECT COUNT(*) FROM aml_fx WHERE ts = TIMESTAMP '{first_minute}'").fetchone()[0]
n_ok = con.execute(f"""
    SELECT COUNT(*) FROM aml_fx a JOIN receiver_features rf ON rf.id = a.id
    WHERE a.ts = TIMESTAMP '{first_minute}'
      AND rf.dst_in_cnt_7d = 0 AND rf.dst_out_cnt_24h = 0
      AND rf.dst_first_seen_in_24h = 0 AND rf.dst_in_sum_usd_7d IS NULL
""").fetchone()[0]
print(f"first-minute: {n_first} | all zero + NULL: {n_ok} | match: {n_first == n_ok}")
assert n_first == n_ok

# --- brute-force on sample_ids ---
mismatches_4c = []
for tx_id in sample_ids:
    dst, ts = con.execute(f"SELECT dst, ts FROM aml_fx WHERE id = {tx_id}").fetchone()
    c = con.execute(f"SELECT * FROM receiver_features WHERE id = {tx_id}").fetchdf().iloc[0]

    for label, h in [("1h", 1), ("24h", 24), ("7d", 168)]:
        cnt, s = con.execute(f"""
            SELECT COUNT(*), SUM(amt_received_usd) FROM aml_fx
            WHERE dst = '{dst}' AND src <> dst AND ts < TIMESTAMP '{ts}'
              AND ts >= TIMESTAMP '{ts}' - INTERVAL '{h} hours'
        """).fetchone()
        s_val = 0.0 if s is None else float(s)
        ok_c = (cnt == c[f"dst_in_cnt_{label}"])
        got = c[f"dst_in_sum_usd_{label}"]
        ok_s = pd.isna(got) if cnt == 0 else (not pd.isna(got) and abs(got - s_val) < 1e-6 * max(1.0, abs(s_val)))
        if not (ok_c and ok_s):
            mismatches_4c.append((tx_id, label, cnt, c[f"dst_in_cnt_{label}"], s_val, got))

    out24 = con.execute(f"""
        SELECT COUNT(*) FROM aml_fx WHERE src = '{dst}' AND src <> dst
          AND ts >= TIMESTAMP '{ts}' - INTERVAL '24 hours' AND ts < TIMESTAMP '{ts}'
    """).fetchone()[0]
    if out24 != c["dst_out_cnt_24h"]:
        mismatches_4c.append((tx_id, "out_24h", out24, c["dst_out_cnt_24h"]))

    fs24 = con.execute(f"""
        WITH w AS (SELECT id, src, ts FROM aml_fx
                   WHERE dst = '{dst}' AND src <> dst
                     AND ts >= TIMESTAMP '{ts}' - INTERVAL '24 hours'
                     AND ts <  TIMESTAMP '{ts}')
        SELECT COUNT(*) FROM w
        WHERE NOT EXISTS (
            SELECT 1 FROM aml_fx p
            WHERE p.dst = '{dst}' AND p.src = w.src
              AND (p.ts < w.ts OR (p.ts = w.ts AND p.id < w.id))
        )
    """).fetchone()[0]
    if fs24 != c["dst_first_seen_in_24h"]:
        mismatches_4c.append((tx_id, "first_seen_24h", fs24, c["dst_first_seen_in_24h"]))

print(f"\nCell 4c brute-force: {len(sample_ids)} rows | mismatches: {len(mismatches_4c)}")
for m in mismatches_4c:
    print(m)
assert len(mismatches_4c) == 0
print("\nCell 4c v2 done: 8 receiver features rebuilt.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[receiver_features v2] wall = 202.4 s | RSS now = 0.35 GB | peak RSS = 9.39 GB
receiver_features rows: 5078345
columns: ['id', 'dst_in_cnt_1h', 'dst_in_cnt_24h', 'dst_in_cnt_7d', 'dst_in_sum_usd_1h', 'dst_in_sum_usd_24h', 'dst_in_sum_usd_7d', 'dst_out_cnt_24h', 'dst_first_seen_in_24h']
     id  dst_in_cnt_1h  dst_in_cnt_24h  dst_in_cnt_7d  dst_in_sum_usd_1h  dst_in_sum_usd_24h  dst_in_sum_usd_7d  dst_out_cnt_24h  dst_first_seen_in_24h
3691341            2.0             5.0           24.0            5300.78             6787.23           38945.79              0.0                    0.0
3792224            0.0             4.0           23.0               0.00             7693.69           39852.25              0.0                    0.0
3792225            0.0             4.0           23.0               0.00             7693.69           39852.25              0.0                    0.0

monotonicity: 0 | new>cnt: 0 | NULL-pattern: 0
cold-start NULL sums (dst_in_sum_usd_7d): 713,911 (14.06%

In [51]:
# Cell 4f v2: final assembly of 34 AML features (#9e sender NULL mask + #9f renamed)
# Explicit column list, no SELECT *, no _1 suffix. Rebuilds aml_features.

# Clean up
con.execute("DROP TABLE IF EXISTS aml_features")

with Timer("4f assemble v2"):
    con.execute("""
        CREATE OR REPLACE TABLE aml_features AS
        WITH hist AS (
            SELECT
                a.id,
                (ou.cum_cnt IS NOT NULL) AS has_any_out,
                (iu.cum_cnt IS NOT NULL) AS has_any_in
            FROM aml_fx a
            ASOF LEFT JOIN cum_out ou ON a.src = ou.account AND ou.ts <= a.ts - INTERVAL '1 minute'
            ASOF LEFT JOIN cum_in  iu ON a.src = iu.account AND iu.ts <= a.ts - INTERVAL '1 minute'
        )
        SELECT
            a.id, a.ts, a.is_laundering, a.src, a.dst,

            -- 12 sender windowed counts (unchanged)
            sf.out_cnt_1h, sf.out_cnt_24h, sf.out_cnt_7d,
            sf.in_cnt_1h,  sf.in_cnt_24h,  sf.in_cnt_7d,

            -- 6 sender sums (NULL-masked per #9e)
            CASE WHEN h.has_any_out THEN sf.out_sum_usd_1h  END AS out_sum_usd_1h,
            CASE WHEN h.has_any_out THEN sf.out_sum_usd_24h END AS out_sum_usd_24h,
            CASE WHEN h.has_any_out THEN sf.out_sum_usd_7d  END AS out_sum_usd_7d,
            CASE WHEN h.has_any_in  THEN sf.in_sum_usd_1h   END AS in_sum_usd_1h,
            CASE WHEN h.has_any_in  THEN sf.in_sum_usd_24h  END AS in_sum_usd_24h,
            CASE WHEN h.has_any_in  THEN sf.in_sum_usd_7d   END AS in_sum_usd_7d,

            -- 2 sender derived, recomputed against masked in_sum_24h
            CASE WHEN sf.in_sum_usd_24h IS NULL OR sf.in_sum_usd_24h = 0 THEN NULL
                 ELSE sf.out_sum_usd_24h / sf.in_sum_usd_24h END AS pass_through_24h,
            sd.mins_since_last_inflow,
            CASE WHEN sf.in_sum_usd_24h IS NULL OR sf.in_sum_usd_24h = 0 THEN NULL
                 ELSE a.amt_paid_usd / sf.in_sum_usd_24h END AS amt_over_inflow_24h,

            -- 2 sender first-seen (#9f)
            sfs.n_first_seen_dst_24h,
            sfs.n_first_seen_senders_in_24h,

            -- 8 receiver (with renamed column per #9f)
            rf.dst_in_cnt_1h, rf.dst_in_cnt_24h, rf.dst_in_cnt_7d,
            rf.dst_in_sum_usd_1h, rf.dst_in_sum_usd_24h, rf.dst_in_sum_usd_7d,
            rf.dst_out_cnt_24h,
            rf.dst_first_seen_in_24h,

            -- 3 pair + ages (self-loop-inclusive from 4d v2)
            pa.pair_cnt_7d,
            pa.src_observed_age_hours,
            pa.dst_observed_age_hours,

            -- 6 transaction-level (from txn_features)
            tf.log_amount_usd, tf.pay_format, tf.pay_ccy,
            tf.is_cross_ccy, tf.hour_of_day, tf.is_self_loop

        FROM aml_fx a
        LEFT JOIN hist                       h   ON h.id  = a.id
        LEFT JOIN sender_features            sf  ON sf.id = a.id
        LEFT JOIN sender_derived_cheap       sd  ON sd.id = a.id
        LEFT JOIN sender_first_seen_24h      sfs ON sfs.id = a.id
        LEFT JOIN receiver_features          rf  ON rf.id = a.id
        LEFT JOIN pair_age_features          pa  ON pa.id = a.id
        LEFT JOIN txn_features               tf  ON tf.id = a.id
    """)

# --- Verify: 34 features, no _1 suffix ---
META = {"id", "ts", "is_laundering", "src", "dst"}
actual = list(con.execute("SELECT * FROM aml_features LIMIT 0").df().columns)
FEATS = [c for c in actual if c not in META]
print(f"\nfeature count: {len(FEATS)} (expected 34)")
print("features:", FEATS)
assert len(FEATS) == 34, f"got {len(FEATS)}"
assert not any(c.endswith("_1") for c in actual), \
    f"suffix collision: {[c for c in actual if c.endswith('_1')]}"

# --- row count ---
n = con.execute("SELECT COUNT(*) FROM aml_features").fetchone()[0]
assert n == 5_078_345, n
print(f"rows: {n}")

# --- post-#9e sender NULL symmetry check ---
print("\n--- sender vs receiver NULL symmetry (per #9e) ---")
sym = con.execute("""
    SELECT
        strftime(ts, '%Y-%m-%d') AS d,
        ROUND(100.0 * SUM(CASE WHEN in_sum_usd_24h     IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS null_sender_in24,
        ROUND(100.0 * SUM(CASE WHEN dst_in_sum_usd_24h IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS null_receiver_in24
    FROM aml_features
    GROUP BY 1 ORDER BY 1
""").fetchdf()
print(sym.to_string(index=False))
print("Expected: both columns high on day 1 (30-60%), declining together.")

# --- split counts ---
print("\n--- split counts ---")
print(con.execute("""
    SELECT split, COUNT(*) n, SUM(is_laundering) n_laund, SUM(in_test_core::INT) core
    FROM aml_features GROUP BY 1 ORDER BY 1
""").fetchdf().to_string(index=False))

# --- first-minute ---
first_minute = con.execute("SELECT MIN(ts) FROM aml_fx").fetchone()[0]
n_first = con.execute(f"SELECT COUNT(*) FROM aml_fx WHERE ts = TIMESTAMP '{first_minute}'").fetchone()[0]
n_ok = con.execute(f"""
    SELECT COUNT(*) FROM aml_features
    WHERE ts = TIMESTAMP '{first_minute}'
      AND out_cnt_1h = 0 AND out_cnt_24h = 0 AND out_cnt_7d = 0
      AND in_cnt_1h = 0 AND in_cnt_24h = 0 AND in_cnt_7d = 0
      AND out_sum_usd_7d IS NULL AND in_sum_usd_7d IS NULL
      AND pair_cnt_7d = 0
      AND src_observed_age_hours IS NULL AND dst_observed_age_hours IS NULL
""").fetchone()[0]
print(f"\nfirst-minute: {n_first} | all zero + NULL: {n_ok} | match: {n_first == n_ok}")
assert n_first == n_ok

# --- save parquet ---
out = PATHS["work"] / "aml_features.parquet"
if out.exists():
    out.unlink()
    print(f"deleted prior parquet")
con.execute(f"COPY aml_features TO '{out.as_posix()}' (FORMAT PARQUET)")
print(f"saved {out} ({out.stat().st_size / 1e6:.1f} MB)")

print("\nCell 4f v2 done: 34 features assembled.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[4f assemble v2] wall = 97.2 s | RSS now = 0.92 GB | peak RSS = 9.39 GB

feature count: 34 (expected 34)
features: ['out_cnt_1h', 'out_cnt_24h', 'out_cnt_7d', 'in_cnt_1h', 'in_cnt_24h', 'in_cnt_7d', 'out_sum_usd_1h', 'out_sum_usd_24h', 'out_sum_usd_7d', 'in_sum_usd_1h', 'in_sum_usd_24h', 'in_sum_usd_7d', 'pass_through_24h', 'mins_since_last_inflow', 'amt_over_inflow_24h', 'n_first_seen_dst_24h', 'n_first_seen_senders_in_24h', 'dst_in_cnt_1h', 'dst_in_cnt_24h', 'dst_in_cnt_7d', 'dst_in_sum_usd_1h', 'dst_in_sum_usd_24h', 'dst_in_sum_usd_7d', 'dst_out_cnt_24h', 'dst_first_seen_in_24h', 'pair_cnt_7d', 'src_observed_age_hours', 'dst_observed_age_hours', 'log_amount_usd', 'pay_format', 'pay_ccy', 'is_cross_ccy', 'hour_of_day', 'is_self_loop']
rows: 5078345

--- sender vs receiver NULL symmetry (per #9e) ---
         d  null_sender_in24  null_receiver_in24
2022-09-01             63.32               56.13
2022-09-02             21.44                7.94
2022-09-03             11.08            

BinderException: Binder Error: Referenced column "split" not found in FROM clause!
Candidate bindings: "pair_cnt_7d", "mins_since_last_inflow", "ts", "in_cnt_1h", "in_cnt_7d"

LINE 2:     SELECT split, COUNT(*) n, SUM(is_laundering) n_laund, SUM(in_test_...
                   ^

In [52]:
# Add split + in_test_core (missing from 4f v2 assembly)
with Timer("add split + in_test_core"):
    con.execute("ALTER TABLE aml_features ADD COLUMN split VARCHAR")
    con.execute("ALTER TABLE aml_features ADD COLUMN in_test_core BOOLEAN")
    con.execute("""
        UPDATE aml_features SET
            split = CASE
                WHEN ts < TIMESTAMP '2022-09-06 13:36' THEN 'train'
                WHEN ts < TIMESTAMP '2022-09-08 16:12' THEN 'valid'
                ELSE 'test'
            END,
            in_test_core = (ts >= TIMESTAMP '2022-09-08 16:12' AND ts < TIMESTAMP '2022-09-11')
    """)

print(con.execute("""
    SELECT split, COUNT(*) n, SUM(is_laundering) n_laund, SUM(in_test_core::INT) core
    FROM aml_features GROUP BY 1 ORDER BY 1
""").fetchdf().to_string(index=False))

[add split + in_test_core] wall = 3.8 s | RSS now = 0.80 GB | peak RSS = 9.39 GB
split       n  n_laund      core
 test 1015882   1798.0 1014774.0
train 3046861   2297.0       0.0
valid 1015602   1082.0       0.0
